In [ ]:
# ==============================================================\n
# R0 — Runtime / DuckDB lock-safe bootstrap\n
# ==============================================================\n
from pathlib import Path\n
import duckdb\n
import tomllib\n
import atexit\n
\n
PROJECT_ROOT = Path.cwd()\n
CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"\n
\n
if CONFIG_TOML_PATH.exists():\n
    with open(CONFIG_TOML_PATH, "rb") as f:\n
        SHARED_CONFIG = tomllib.load(f)\n
    print("Loaded shared config:", CONFIG_TOML_PATH)\n
else:\n
    SHARED_CONFIG = {\n
        "paths": {\n
            "data_dir": "content",\n
            "output_dir": "outputs_benchmark_survival",\n
            "tables_subdir": "tables",\n
            "metadata_subdir": "metadata",\n
            "data_output_subdir": "data",\n
            "duckdb_filename": "benchmark_survival.duckdb",\n
        },\n
        "benchmark": {\n
            "seed": 42,\n
            "test_size": 0.30,\n
            "early_window_weeks": 4,\n
            "main_enrollment_window_weeks": 4,\n
        },\n
    }\n
    print("Shared config TOML not found. Using defaults in-memory.")\n
\n
paths_cfg = SHARED_CONFIG.get("paths", {})\n
SHARED_BENCHMARK_CONFIG = SHARED_CONFIG.get("benchmark", {})\n
\n
def _resolve_project_path(raw_path: str) -> Path:\n
    p = Path(raw_path)\n
    return p if p.is_absolute() else PROJECT_ROOT / p\n
\n
DATA_DIR = _resolve_project_path(paths_cfg.get("data_dir", "content"))\n
OUTPUT_DIR = _resolve_project_path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))\n
TABLES_DIR = OUTPUT_DIR / paths_cfg.get("tables_subdir", "tables")\n
METADATA_DIR = OUTPUT_DIR / paths_cfg.get("metadata_subdir", "metadata")\n
DATA_OUTPUT_DIR = OUTPUT_DIR / paths_cfg.get("data_output_subdir", "data")\n
DUCKDB_PATH = OUTPUT_DIR / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")\n
\n
for p in [OUTPUT_DIR, TABLES_DIR, METADATA_DIR, DATA_OUTPUT_DIR]:\n
    p.mkdir(parents=True, exist_ok=True)\n
\n
if "con" in globals():\n
    try:\n
        con.close()\n
        print("Closed previous DuckDB connection before reconnecting.")\n
    except Exception as _close_err:\n
        print(f"Warning while closing previous DuckDB connection: {_close_err}")\n
\n
con = duckdb.connect(str(DUCKDB_PATH))\n
\n
def _close_duckdb_connection() -> None:\n
    if "con" in globals():\n
        try:\n
            con.close()\n
            print("DuckDB connection closed.")\n
        except Exception:\n
            pass\n
\n
if "_duckdb_close_registered" not in globals():\n
    atexit.register(_close_duckdb_connection)\n
    _duckdb_close_registered = True\n
\n
print("Runtime context ready.")\n
print("- OUTPUT_DIR  :", OUTPUT_DIR)\n
print("- DUCKDB_PATH :", DUCKDB_PATH)\n

# D — Modeling

Scope:
- benchmark protocol
- model-specific preprocessing
- performance-oriented tuned models
- consolidated benchmark comparison
- expanded calibration interpretation


## D1 — Define Benchmark Metrics and Evaluation Protocol (Revised v3)


In [ ]:
# ==============================================================
# D1 — Define Benchmark Metrics and Evaluation Protocol (Revised v3)
# --------------------------------------------------------------
# Purpose:
#   Define the formal benchmark evaluation protocol for the study,
#   including:
#     - metric hierarchy
#     - official benchmark horizons
#     - common unit of comparison
#     - censoring treatment
#     - concordance convention
#     - dynamic-vs-static prediction rules
#     - calibration protocol, including strengthened diagnostics
#
# Methodological note:
#   This benchmark is intentionally survival-oriented. The goal is
#   not to compare models using a single classification score, but
#   to assess them under a unified time-to-event evaluation frame.
#
#   This revised version makes explicit:
#     - how censoring is handled in Brier / IBS,
#     - which concordance convention is used,
#     - how dynamic predictions are defined for discrete-time models,
#     - how leakage is prevented,
#     - how horizon-level calibration is computed and reported,
#     - and how the expanded calibration diagnostics from Group E
#       fit into the benchmark protocol.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#
# Optional inputs:
#   - table_p26_4_expanded_calibration_protocol.csv
#   - table_p26_5_paper_summary_all_tuned_models.csv
#   - table_p26_6_calibration_model_rankings.csv
#
# Outputs:
#   - BENCHMARK_CONFIG dictionary
#   - benchmark_config.json
#   - table_benchmark_metrics_manifest.csv
#   - table_evaluation_protocol_audit.csv
# ==============================================================

print("\n" + "=" * 70)
print("D1 — Define Benchmark Metrics and Evaluation Protocol (Revised v3)")
print("=" * 70)
print("Methodological note: the benchmark is survival-oriented and")
print("will be evaluated under a common enrollment-level protocol.")
print("This revision makes censoring, concordance, prediction timing,")
print("and the expanded calibration protocol explicit.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

OUTPUT_DIR = Path(OUTPUT_DIR)
TABLES_DIR = Path(TABLES_DIR)
METADATA_DIR = Path(METADATA_DIR)

# ------------------------------
# 2) Detect official horizons
# ------------------------------
p26_4_protocol_path = TABLES_DIR / "table_p26_4_expanded_calibration_protocol.csv"

if p26_4_protocol_path.exists():
    p26_4_protocol_df = pd.read_csv(p26_4_protocol_path)
    shared_horizons_row = p26_4_protocol_df.loc[
        p26_4_protocol_df["protocol_component"] == "shared_horizons",
        "value"
    ]
    if not shared_horizons_row.empty:
        BENCHMARK_HORIZONS = [
            int(x.strip())
            for x in str(shared_horizons_row.iloc[0]).split(",")
            if str(x).strip()
        ]
    else:
        BENCHMARK_HORIZONS = [10, 20, 30]
else:
    BENCHMARK_HORIZONS = [10, 20, 30]

BENCHMARK_HORIZONS = sorted(list(dict.fromkeys(BENCHMARK_HORIZONS)))

# ------------------------------
# 3) Protocol core definitions
# ------------------------------
BENCHMARK_CONFIG = {
    "protocol_name": "harmonized_survival_benchmark",
    "protocol_version": "P15_revised_v3",
    "created_at": datetime.utcnow().isoformat() + "Z",

    # Unit and event frame
    "unit_of_analysis": "enrollment",
    "event_definition": "withdrawn_with_valid_date_unregistration",
    "time_granularity": "weekly",
    "official_benchmark_horizons": BENCHMARK_HORIZONS,

    # Family comparison frame
    "comparison_scope": "cross-family survival-oriented benchmark",
    "comparison_unit_for_reporting": "enrollment-level horizon-specific predictions",
    "comparable_reporting_horizons": BENCHMARK_HORIZONS,

    # Prediction regimes
    "prediction_regimes": {
        "continuous_time_comparable_arm": (
            "Enrollment-level static prediction using early-window summarized covariates, "
            "evaluated at shared fixed horizons."
        ),
        "discrete_time_dynamic_arm": (
            "Weekly person-period prediction with horizon-level aggregation to enrollment-level "
            "risk summaries at shared fixed horizons."
        ),
        "benchmark_rule": (
            "Cross-family comparison is performed only at shared fixed horizons under a common "
            "enrollment-level reporting frame."
        ),
    },

    # Leakage / train-test discipline
    "anti_leakage_rules": [
        "All learned preprocessing transformations are fit on training data only.",
        "Test data are transformed using the training-fitted objects without re-estimation.",
        "Model tuning and internal validation remain restricted to training-side information.",
        "Evaluation uses only test-set predictions generated after model freeze."
    ],

    # Censoring conventions
    "censoring_protocol": {
        "event_type": "withdrawn",
        "right_censoring": True,
        "censoring_definition": (
            "Enrollments without an observed withdrawal event by the end of follow-up are treated as right-censored."
        ),
        "brier_and_ibs_treatment": (
            "Brier-type metrics and integrated Brier score follow the benchmark-specific censoring-aware convention "
            "already implemented in the notebook outputs."
        ),
    },

    # Concordance conventions
    "concordance_protocol": {
        "primary_time_dependent_concordance": (
            "Time-dependent concordance under the benchmark-specific implementation."
        ),
        "secondary_horizon_specific_discrimination": (
            "AUC or horizon-specific discrimination summaries evaluated at the shared horizons."
        ),
    },

    # Calibration conventions
    "calibration_protocol": {
        "primary_calibration_metric": "weighted_absolute_calibration_gap_by_horizon",
        "primary_metric_definition": (
            "Sample-size-weighted absolute gap between mean predicted risk and observed event frequency across calibration bins."
        ),
        "primary_metric_role": (
            "Primary calibration ranking signal in the tuned cross-model comparison."
        ),
        "supporting_metrics": [
            "brier_at_horizon",
            "event_rate_at_horizon",
            "support_n_at_horizon"
        ],
        "expanded_strengthening_metrics": [
            "approx_calibration_intercept_by_horizon",
            "approx_calibration_slope_by_horizon"
        ],
        "strengthening_metric_interpretation": (
            "Intercept and slope are used as calibration-strengthening diagnostics and interpretive qualifiers, "
            "not as a standalone replacement for the primary calibration gap."
        ),
        "optional_metric_not_required": "ici_style_summary",
        "official_horizons": BENCHMARK_HORIZONS,
        "comparability_rule": (
            "Calibration comparison is restricted to shared horizons where all compared tuned models provide valid outputs."
        ),
    },

    # Metric hierarchy
    "metric_hierarchy": {
        "primary_global_metrics": [
            "integrated_brier_score",
            "time_dependent_concordance"
        ],
        "primary_horizon_metrics": [
            "brier_at_horizon",
            "weighted_absolute_calibration_gap_by_horizon"
        ],
        "secondary_metrics": [
            "horizon_specific_auc",
            "predicted_vs_observed_survival"
        ],
        "contextual_audit_metrics": [
            "support_n_at_horizon",
            "event_rate_at_horizon",
            "approx_calibration_intercept_by_horizon",
            "approx_calibration_slope_by_horizon"
        ],
    },

    # Sensitivity / robustness framing
    "sensitivity_design": {
        "early_window_length_sensitivity": [2, 4, 6],
        "horizon_choice_stress_test": BENCHMARK_HORIZONS,
        "ablation_available": True,
        "note": (
            "Window choice and horizon choice are treated as explicit sensitivity dimensions rather than hidden defaults."
        ),
    },
}

# ------------------------------
# 4) Benchmark metrics manifest
# ------------------------------
metrics_manifest_rows = [
    {
        "metric_name": "integrated_brier_score",
        "metric_family": "overall_prediction_quality",
        "metric_role": "primary_global",
        "lower_is_better": True,
        "time_scope": "integrated_over_followup",
        "used_in_primary_ranking": True,
        "interpretation": "Integrated Brier Score under the benchmark censoring-aware convention."
    },
    {
        "metric_name": "time_dependent_concordance",
        "metric_family": "discrimination",
        "metric_role": "primary_global",
        "lower_is_better": False,
        "time_scope": "integrated_over_followup",
        "used_in_primary_ranking": True,
        "interpretation": "Time-dependent concordance under the benchmark convention."
    },
    {
        "metric_name": "brier_at_horizon",
        "metric_family": "overall_prediction_quality",
        "metric_role": "primary_horizon",
        "lower_is_better": True,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": True,
        "interpretation": "Horizon-specific Brier score reported at the official benchmark horizons."
    },
    {
        "metric_name": "weighted_absolute_calibration_gap_by_horizon",
        "metric_family": "calibration",
        "metric_role": "primary_horizon",
        "lower_is_better": True,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": True,
        "interpretation": "Primary calibration ranking signal based on weighted absolute gap across bins."
    },
    {
        "metric_name": "horizon_specific_auc",
        "metric_family": "discrimination",
        "metric_role": "secondary",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Secondary discrimination metric reported at shared horizons."
    },
    {
        "metric_name": "support_n_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "contextual",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Number of evaluable enrollments contributing to each horizon."
    },
    {
        "metric_name": "event_rate_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "contextual",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Observed event prevalence at each shared benchmark horizon."
    },
    {
        "metric_name": "approx_calibration_intercept_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "diagnostic",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Calibration-strengthening diagnostic for directional bias at each horizon."
    },
    {
        "metric_name": "approx_calibration_slope_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "diagnostic",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons",
        "used_in_primary_ranking": False,
        "interpretation": "Calibration-strengthening diagnostic for compression/dispersion of predictions."
    },
    {
        "metric_name": "predicted_vs_observed_survival",
        "metric_family": "diagnostic",
        "metric_role": "diagnostic",
        "lower_is_better": False,
        "time_scope": "shared_fixed_horizons_or_curve_level",
        "used_in_primary_ranking": False,
        "interpretation": "Visual agreement between predicted and observed survival summaries."
    },
]

benchmark_metrics_manifest_df = pd.DataFrame(metrics_manifest_rows)

# ------------------------------
# 5) Protocol audit table
# ------------------------------
protocol_audit_rows = [
    {
        "component": "unit_of_analysis",
        "status": "defined",
        "protocol_value": BENCHMARK_CONFIG["unit_of_analysis"],
        "notes": "Common unit for cross-family reporting."
    },
    {
        "component": "event_definition",
        "status": "defined",
        "protocol_value": BENCHMARK_CONFIG["event_definition"],
        "notes": "Observed withdrawal event with valid time stamp."
    },
    {
        "component": "official_horizons",
        "status": "defined",
        "protocol_value": ",".join(map(str, BENCHMARK_HORIZONS)),
        "notes": "Shared benchmark horizons used for fixed-horizon comparison."
    },
    {
        "component": "dynamic_vs_static_prediction_rule",
        "status": "defined",
        "protocol_value": "shared_horizon_enrollment_level_reporting",
        "notes": "Different model families are compared only after horizon-level alignment."
    },
    {
        "component": "censoring_treatment",
        "status": "defined",
        "protocol_value": "right_censoring_enabled",
        "notes": "Brier/IBS follow the benchmark-specific censoring-aware implementation."
    },
    {
        "component": "primary_calibration_metric",
        "status": "defined",
        "protocol_value": "weighted_absolute_calibration_gap_by_horizon",
        "notes": "Primary calibration ranking signal."
    },
    {
        "component": "expanded_calibration_strengthening",
        "status": "defined",
        "protocol_value": "intercept_and_slope_by_horizon",
        "notes": "Strengthening diagnostics added after Group E."
    },
    {
        "component": "ici_style_metric",
        "status": "optional_not_required",
        "protocol_value": "not_canonical_for_core_benchmark",
        "notes": "Kept optional; not required for the benchmark core."
    },
    {
        "component": "sensitivity_window_length",
        "status": "defined",
        "protocol_value": "2_4_6",
        "notes": "Early-window length treated as explicit sensitivity dimension."
    },
    {
        "component": "sensitivity_horizon_choice",
        "status": "defined",
        "protocol_value": ",".join(map(str, BENCHMARK_HORIZONS)),
        "notes": "Horizon choice treated as explicit stress test dimension."
    },
]

evaluation_protocol_audit_df = pd.DataFrame(protocol_audit_rows)

# ------------------------------
# 6) Optional linkage to P26 outputs
# ------------------------------
p26_5_summary_path = TABLES_DIR / "table_p26_5_paper_summary_all_tuned_models.csv"
p26_6_rankings_path = TABLES_DIR / "table_p26_6_calibration_model_rankings.csv"

if p26_5_summary_path.exists():
    p26_5_summary_df = pd.read_csv(p26_5_summary_path)
    if not p26_5_summary_df.empty:
        best_calibration_model = p26_5_summary_df.sort_values(
            by=["mean_calib_gap", "mean_brier"],
            ascending=[True, True]
        ).iloc[0]
        BENCHMARK_CONFIG["current_calibration_leader_from_p26_5"] = {
            "model_id": str(best_calibration_model["model_id"]),
            "model": str(best_calibration_model["model"]),
            "mean_calib_gap": float(best_calibration_model["mean_calib_gap"]),
            "mean_brier": float(best_calibration_model["mean_brier"]),
        }

if p26_6_rankings_path.exists():
    p26_6_rankings_df = pd.read_csv(p26_6_rankings_path)
    if not p26_6_rankings_df.empty:
        top_ranked = p26_6_rankings_df.sort_values(
            by="rank_overall_primary",
            ascending=True
        ).iloc[0]
        BENCHMARK_CONFIG["current_primary_calibration_ranking_leader_from_p26_6"] = {
            "model_id": str(top_ranked["model_id"]),
            "model": str(top_ranked["model"]),
            "rank_overall_primary": int(top_ranked["rank_overall_primary"]),
            "overall_calibration_reading": str(top_ranked["overall_calibration_reading"]),
        }

# ------------------------------
# 7) Save outputs
# ------------------------------
benchmark_config_path = METADATA_DIR / "benchmark_config.json"
metrics_manifest_path = TABLES_DIR / "table_benchmark_metrics_manifest.csv"
protocol_audit_path = TABLES_DIR / "table_evaluation_protocol_audit.csv"

benchmark_metrics_manifest_df.to_csv(metrics_manifest_path, index=False)
evaluation_protocol_audit_df.to_csv(protocol_audit_path, index=False)
save_json(BENCHMARK_CONFIG, benchmark_config_path)

# ------------------------------
# 8) Display
# ------------------------------
print("\nBenchmark configuration (core fields):")
display(pd.DataFrame([
    {"key": "protocol_name", "value": BENCHMARK_CONFIG["protocol_name"]},
    {"key": "protocol_version", "value": BENCHMARK_CONFIG["protocol_version"]},
    {"key": "unit_of_analysis", "value": BENCHMARK_CONFIG["unit_of_analysis"]},
    {"key": "event_definition", "value": BENCHMARK_CONFIG["event_definition"]},
    {"key": "official_benchmark_horizons", "value": ",".join(map(str, BENCHMARK_CONFIG["official_benchmark_horizons"]))},
    {"key": "primary_calibration_metric", "value": BENCHMARK_CONFIG["calibration_protocol"]["primary_calibration_metric"]},
    {"key": "strengthening_metrics", "value": ", ".join(BENCHMARK_CONFIG["calibration_protocol"]["expanded_strengthening_metrics"])},
]))

print("\nBenchmark metrics manifest:")
display(benchmark_metrics_manifest_df)

print("\nEvaluation protocol audit:")
display(evaluation_protocol_audit_df)

print("\nSaved:")
print("-", benchmark_config_path.resolve())
print("-", metrics_manifest_path.resolve())
print("-", protocol_audit_path.resolve())

### D1.1 — Materialize Benchmark Runtime Aliases

In [ ]:
# ==============================================================
# D1.1 — Materialize Benchmark Runtime Aliases
# --------------------------------------------------------------
# Purpose:
#   Recreate the benchmark constants expected by downstream cells
#   under sequential execution after the protocol cell has run.
#
# Methodological note:
#   This cell does not recompute metrics.
#   It only materializes compatibility aliases and a guarded
#   calibration-bin constant from the canonical benchmark protocol.
# ==============================================================

print("\n" + "=" * 70)
print("D1.1 — Materialize Benchmark Runtime Aliases")
print("=" * 70)
print("Methodological note: this cell materializes benchmark constants only.")

required_names = ["BENCHMARK_CONFIG", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run D1 first."
    )

if "BENCHMARK_HORIZONS" in globals():
    detected_horizons = [int(h) for h in BENCHMARK_HORIZONS]
else:
    detected_horizons = [
        int(h)
        for h in BENCHMARK_CONFIG.get("official_benchmark_horizons", [10, 20, 30])
    ]

HORIZONS_WEEKS = sorted(list(dict.fromkeys(detected_horizons)))
HORIZON_WEEKS = list(HORIZONS_WEEKS)
BENCHMARK_HORIZONS = list(HORIZONS_WEEKS)

calibration_protocol = BENCHMARK_CONFIG.setdefault("calibration_protocol", {})
calibration_bin_candidates = [
    calibration_protocol.get("n_bins"),
    calibration_protocol.get("target_n_bins"),
    calibration_protocol.get("calibration_bins"),
    BENCHMARK_CONFIG.get("calibration_bins"),
]

CALIBRATION_BINS = next(
    (
        int(value)
        for value in calibration_bin_candidates
        if value is not None and str(value).strip() != ""
    ),
    10,
)

BENCHMARK_CONFIG["official_benchmark_horizons"] = list(HORIZONS_WEEKS)
BENCHMARK_CONFIG["comparable_reporting_horizons"] = list(HORIZONS_WEEKS)
calibration_protocol["official_horizons"] = list(HORIZONS_WEEKS)
calibration_protocol["target_n_bins"] = int(CALIBRATION_BINS)
BENCHMARK_CONFIG["calibration_bins"] = int(CALIBRATION_BINS)

benchmark_config_path = METADATA_DIR / "benchmark_config.json"
save_json(BENCHMARK_CONFIG, benchmark_config_path)

print("\nMaterialized runtime aliases:")
print("- HORIZONS_WEEKS:", HORIZONS_WEEKS)
print("- HORIZON_WEEKS:", HORIZON_WEEKS)
print("- CALIBRATION_BINS:", CALIBRATION_BINS)
print("\nSaved:")
print("-", benchmark_config_path.resolve())

## D2 — Treatment for the Linear Discrete-Time Hazard Model


In [ ]:
# ==============================================================
# D2 — Treatment for the Linear Discrete-Time Hazard Model
# CORRECTED VERSION (robust to canonical-vs-materialized feature names)
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the linear
#   discrete-time hazard baseline, with all learned transformations
#   fitted on training data only.
#
# Methodological note:
#   This step performs model-specific preprocessing for the linear
#   discrete-time hazard baseline. The following learned operations
#   are fit on training data only:
#     - numeric imputation
#     - categorical imputation
#     - one-hot encoding
#     - numeric standardization
#
#   The same fitted preprocessing object is then applied to the
#   test data without re-estimating any parameter, preventing leakage.
#
# Design note:
#   After the benchmark-architecture refactor, P10 started exposing
#   a canonical minimal dynamic-arm config (e.g., "total_clicks"),
#   while the materialized person-period tables still use the richer
#   notebook-native names (e.g., "total_clicks_week").
#   This cell therefore resolves canonical aliases to the actual
#   materialized columns instead of requiring exact name identity.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - TEMPORAL_FEATURES_DISCRETE
#
# Outputs:
#   - X_train_linear, X_test_linear
#   - y_train_linear, y_test_linear
#   - linear_preprocessor
#   - linear_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D2 — Treatment for the Linear Discrete-Time Hazard Model")
print("=" * 70)
print("Methodological note: all learned preprocessing transformations")
print("are fit on training data only and then applied to test data.")

# ------------------------------
# 1) Dependency checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "TEMPORAL_FEATURES_DISCRETE"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df = con.execute("SELECT * FROM pp_linear_hazard_ready_train").fetchdf()
test_df = con.execute("SELECT * FROM pp_linear_hazard_ready_test").fetchdf()

if train_df.empty:
    raise ValueError("pp_linear_hazard_ready_train is empty.")
if test_df.empty:
    raise ValueError("pp_linear_hazard_ready_test is empty.")

# ------------------------------
# 3) Define target and operational feature columns
# ------------------------------
target_col = "event_t"

# These are the notebook-native operational columns currently
# materialized in pp_linear_hazard_input / ready tables.
categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "week",
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]

linear_feature_columns = categorical_features + numeric_features

# ------------------------------
# 4) Canonical-to-materialized alias resolution
# ------------------------------
# P10 may expose a canonical minimal config such as:
#   TEMPORAL_FEATURES_DISCRETE = ["week", "total_clicks"]
# while the actual discrete tables use "total_clicks_week".
feature_alias_map = {
    "total_clicks": "total_clicks_week",
}

expected_features_raw = list(STATIC_FEATURES) + list(TEMPORAL_FEATURES_DISCRETE)
expected_features_resolved = [
    feature_alias_map.get(col, col) for col in expected_features_raw
]

# Require only that the canonical configured features are covered
# by the operational treatment spec after alias resolution.
missing_from_defined = [
    c for c in expected_features_resolved
    if c not in linear_feature_columns
]

# Also verify that the operational columns truly exist in both splits.
missing_in_train = [c for c in linear_feature_columns if c not in train_df.columns]
missing_in_test = [c for c in linear_feature_columns if c not in test_df.columns]

if missing_from_defined:
    raise ValueError(
        "Configured canonical features are not covered by the operational "
        f"linear treatment spec after alias resolution: {missing_from_defined}. "
        f"Raw expected features were: {expected_features_raw}"
    )

if missing_in_train or missing_in_test:
    raise ValueError(
        "Operational linear feature columns are missing from the materialized "
        "train/test tables.\n"
        f"Missing in train: {missing_in_train}\n"
        f"Missing in test : {missing_in_test}"
    )

if target_col not in train_df.columns or target_col not in test_df.columns:
    raise ValueError(
        f"Target column '{target_col}' is missing from train/test tables."
    )

# ------------------------------
# 5) Split X / y
# ------------------------------
X_train_raw = train_df[linear_feature_columns].copy()
X_test_raw = test_df[linear_feature_columns].copy()

y_train_linear = train_df[target_col].astype(int).copy()
y_test_linear = test_df[target_col].astype(int).copy()

# ------------------------------
# 6) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 7) Fit on train only, transform train/test
# ------------------------------
X_train_linear = linear_preprocessor.fit_transform(X_train_raw)
X_test_linear = linear_preprocessor.transform(X_test_raw)

linear_feature_names_out = linear_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 8) Audit tables
# ------------------------------
preproc_audit_df = pd.DataFrame({
    "component": [
        "target_column",
        "n_train_rows",
        "n_test_rows",
        "n_operational_features",
        "n_numeric_features",
        "n_categorical_features",
        "n_output_features_after_preprocessing",
        "train_event_rate",
        "test_event_rate",
    ],
    "value": [
        target_col,
        int(len(train_df)),
        int(len(test_df)),
        int(len(linear_feature_columns)),
        int(len(numeric_features)),
        int(len(categorical_features)),
        int(len(linear_feature_names_out)),
        float(y_train_linear.mean()),
        float(y_test_linear.mean()),
    ],
})

feature_manifest_df = pd.DataFrame({
    "feature_name_operational": linear_feature_columns,
    "feature_role": [
        "categorical" if c in categorical_features else "numeric"
        for c in linear_feature_columns
    ],
    "present_in_train": [c in train_df.columns for c in linear_feature_columns],
    "present_in_test": [c in test_df.columns for c in linear_feature_columns],
    "covered_by_canonical_config_after_alias": [
        c in expected_features_resolved for c in linear_feature_columns
    ],
    "canonical_source_name": [
        next(
            (raw for raw, resolved in zip(expected_features_raw, expected_features_resolved) if resolved == c),
            ""
        )
        for c in linear_feature_columns
    ],
})

canonical_alignment_df = pd.DataFrame({
    "canonical_feature_raw": expected_features_raw,
    "canonical_feature_resolved": expected_features_resolved,
    "covered_by_operational_treatment_spec": [
        c in linear_feature_columns for c in expected_features_resolved
    ],
})

output_feature_manifest_df = pd.DataFrame({
    "preprocessed_feature_name": linear_feature_names_out
})

# ------------------------------
# 9) Save outputs
# ------------------------------
preproc_audit_path = TABLES_DIR / "table_p16_linear_treatment_audit.csv"
feature_manifest_path = TABLES_DIR / "table_p16_linear_feature_manifest.csv"
canonical_alignment_path = TABLES_DIR / "table_p16_linear_canonical_alignment.csv"
output_manifest_path = TABLES_DIR / "table_p16_linear_output_feature_manifest.csv"
metadata_path = METADATA_DIR / "metadata_p16_linear_treatment.json"

preproc_audit_df.to_csv(preproc_audit_path, index=False)
feature_manifest_df.to_csv(feature_manifest_path, index=False)
canonical_alignment_df.to_csv(canonical_alignment_path, index=False)
output_feature_manifest_df.to_csv(output_manifest_path, index=False)

save_json({
    "step": "P16",
    "model_family": "linear_discrete_time_hazard",
    "train_table": "pp_linear_hazard_ready_train",
    "test_table": "pp_linear_hazard_ready_test",
    "target_column": target_col,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "operational_feature_columns": linear_feature_columns,
    "canonical_expected_features_raw": expected_features_raw,
    "canonical_expected_features_resolved": expected_features_resolved,
    "feature_alias_map": feature_alias_map,
    "n_train_rows": int(len(train_df)),
    "n_test_rows": int(len(test_df)),
    "n_output_features_after_preprocessing": int(len(linear_feature_names_out)),
    "train_event_rate": float(y_train_linear.mean()),
    "test_event_rate": float(y_test_linear.mean()),
    "methodological_note": (
        "All learned preprocessing operations were fit on training data only "
        "and then applied unchanged to test data."
    ),
    "design_note": (
        "Canonical config features were resolved to notebook-native materialized "
        "column names before validation."
    ),
}, metadata_path)

# ------------------------------
# 10) Console summary
# ------------------------------
print("\nOperational feature columns used by the linear model:")
for col in linear_feature_columns:
    print(" -", col)

print("\nCanonical config alignment:")
display(canonical_alignment_df)

print("\nTreatment audit:")
display(preproc_audit_df)

print("\nSaved:")
print("-", preproc_audit_path.resolve())
print("-", feature_manifest_path.resolve())
print("-", canonical_alignment_path.resolve())
print("-", output_manifest_path.resolve())
print("-", metadata_path.resolve())

### D2.1 — Performance-Oriented Tuning for the Linear Discrete-Time Hazard Model (Revised)


### Funcao: get_pred_at_horizon

Definicao isolada de get_pred_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D2.1 — Performance-Oriented Tuning for the Linear Discrete-Time Hazard Model (Revised)
# --------------------------------------------------------------
# Purpose:
#   Perform a controlled hyperparameter search for the linear
#   discrete-time hazard model, select the best candidate, refit
#   it on the full training set, and evaluate it under the same
#   survival-oriented benchmark protocol defined in D1.
#
# Methodological note:
#   This revised version preserves the tuning step but restores
#   full benchmark comparability by computing:
#     - IBS
#     - C-index
#     - horizon-specific Brier score
#     - calibration at horizons
#     - risk AUC at horizons
#     - predicted vs observed survival
#
#   The model remains a discrete-time hazard model:
#   weekly probabilities are interpreted as hazards and enrollment-
#   level survival is reconstructed from hazard trajectories.
# ==============================================================

print("\n" + "=" * 70)
print("D2.1 — Performance-Oriented Tuning for the Linear Discrete-Time Hazard Model (Revised)")
print("=" * 70)
print("Methodological note: this step performs controlled tuning and")
print("evaluates the best candidate under the same survival-oriented")
print("benchmark protocol defined in D1.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_linear", "X_test_linear",
    "y_train_linear", "y_test_linear",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.model_selection import GroupShuffleSplit

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for the revised P17.2 evaluation.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Resolve source dataframes
# ------------------------------
if "train_df_linear" in globals():
    train_df_linear_local = train_df_linear.copy()
elif "pp_linear_hazard_ready_train" in globals():
    train_df_linear_local = pp_linear_hazard_ready_train.copy()
elif "con" in globals():
    train_df_linear_local = con.execute("SELECT * FROM pp_linear_hazard_ready_train").fetchdf()
else:
    raise NameError("Could not resolve train_df_linear / pp_linear_hazard_ready_train.")

if "test_df_linear" in globals():
    test_df_linear_local = test_df_linear.copy()
elif "pp_linear_hazard_ready_test" in globals():
    test_df_linear_local = pp_linear_hazard_ready_test.copy()
elif "con" in globals():
    test_df_linear_local = con.execute("SELECT * FROM pp_linear_hazard_ready_test").fetchdf()
else:
    raise NameError("Could not resolve test_df_linear / pp_linear_hazard_ready_test.")

if "linear_preprocessor" in globals():
    linear_preprocessor_obj = linear_preprocessor
elif "preprocessor_linear" in globals():
    linear_preprocessor_obj = preprocessor_linear
else:
    linear_preprocessor_obj = None

# ------------------------------
# 4) Train/validation split at enrollment level
# ------------------------------
groups = train_df_linear_local["enrollment_id"].astype(str).to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
subtrain_idx, val_idx = next(gss.split(X_train_linear, y_train_linear, groups=groups))

X_subtrain = X_train_linear[subtrain_idx]
X_val = X_train_linear[val_idx]
y_subtrain = y_train_linear[subtrain_idx]
y_val = y_train_linear[val_idx]

# ------------------------------
# 5) Tuning config
# ------------------------------
LINEAR_TUNING_CONFIG = {
    "model_name": "linear_discrete_time_hazard_tuned",
    "search_space": {
        "penalty": ["l1", "l2"],
        "C": [0.01, 0.1, 1.0, 10.0],
    },
    "solver": "liblinear",
    "class_weight": None,
    "max_iter": 2000,
    "selection_metric": "val_log_loss",
    "benchmark_positioning_note": (
        "Performance-oriented tuned linear discrete-time hazard benchmark."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

# ------------------------------
# 6) Run tuning grid
# ------------------------------
candidate_rows = []
candidate_models = []

candidate_id = 0
for penalty, C in itertools.product(
    LINEAR_TUNING_CONFIG["search_space"]["penalty"],
    LINEAR_TUNING_CONFIG["search_space"]["C"]
):
    candidate_id += 1

    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver=LINEAR_TUNING_CONFIG["solver"],
        class_weight=LINEAR_TUNING_CONFIG["class_weight"],
        max_iter=LINEAR_TUNING_CONFIG["max_iter"],
        random_state=RANDOM_SEED,
    )

    model.fit(X_subtrain, y_subtrain)
    val_pred = np.clip(model.predict_proba(X_val)[:, 1], 1e-8, 1 - 1e-8)

    val_logloss = log_loss(y_val, val_pred, labels=[0, 1])
    val_brier = brier_score_loss(y_val, val_pred)

    if len(np.unique(y_val)) >= 2:
        val_roc_auc = roc_auc_score(y_val, val_pred)
        val_pr_auc = average_precision_score(y_val, val_pred)
    else:
        val_roc_auc = np.nan
        val_pr_auc = np.nan

    candidate_rows.append({
        "candidate_id": candidate_id,
        "penalty": penalty,
        "C": C,
        "val_log_loss": float(val_logloss),
        "val_brier": float(val_brier),
        "val_roc_auc": float(val_roc_auc) if pd.notna(val_roc_auc) else np.nan,
        "val_pr_auc": float(val_pr_auc) if pd.notna(val_pr_auc) else np.nan,
    })

tuning_results_df = pd.DataFrame(candidate_rows).sort_values(
    by=["val_log_loss", "val_brier", "candidate_id"],
    ascending=[True, True, True]
).reset_index(drop=True)

best_row = tuning_results_df.iloc[0]
best_candidate_df = pd.DataFrame([best_row.to_dict()])

best_penalty = best_row["penalty"]
best_C = float(best_row["C"])

# ------------------------------
# 7) Refit best model on full train
# ------------------------------
best_model = LogisticRegression(
    penalty=best_penalty,
    C=best_C,
    solver=LINEAR_TUNING_CONFIG["solver"],
    class_weight=LINEAR_TUNING_CONFIG["class_weight"],
    max_iter=LINEAR_TUNING_CONFIG["max_iter"],
    random_state=RANDOM_SEED,
)
best_model.fit(X_train_linear, y_train_linear)

# ------------------------------
# 8) Predict row-level hazards on test
# ------------------------------
test_pred_row = np.clip(best_model.predict_proba(X_test_linear)[:, 1], 1e-8, 1 - 1e-8)

test_pred_df = test_df_linear_local.copy()
test_pred_df["pred_hazard"] = test_pred_row

required_cols = ["enrollment_id", "week", "event_t"]
missing_eval_cols = [c for c in required_cols if c not in test_pred_df.columns]
if missing_eval_cols:
    raise ValueError(f"Missing required columns in test dataframe: {missing_eval_cols}")

test_pred_df = test_pred_df.sort_values(["enrollment_id", "week"]).copy()
test_pred_df["pred_survival"] = test_pred_df.groupby("enrollment_id")["pred_hazard"].transform(
    lambda s: (1.0 - s).cumprod()
)
test_pred_df["pred_risk"] = 1.0 - test_pred_df["pred_survival"]

# enrollment-level truth
truth_test = (
    test_pred_df.groupby("enrollment_id", as_index=False)
    .agg(
        event=("event_observed", "max") if "event_observed" in test_pred_df.columns else ("event_t", "max"),
        duration=("time_for_split", "max") if "time_for_split" in test_pred_df.columns else ("week", "max"),
    )
)

truth_train = (
    train_df_linear_local.groupby("enrollment_id", as_index=False)
    .agg(
        event=("event_observed", "max") if "event_observed" in train_df_linear_local.columns else ("event_t", "max"),
        duration=("time_for_split", "max") if "time_for_split" in train_df_linear_local.columns else ("week", "max"),
    )
)

durations_test = truth_test["duration"].astype(int).to_numpy()
events_test = truth_test["event"].astype(int).to_numpy()

# ------------------------------
# 9) Build survival matrix for pycox evaluation
# ------------------------------
surv_wide = (
    test_pred_df[["enrollment_id", "week", "pred_survival"]]
    .drop_duplicates(subset=["enrollment_id", "week"])
    .pivot(index="week", columns="enrollment_id", values="pred_survival")
    .sort_index()
)

max_week_test = int(test_pred_df["week"].max())
full_week_index = pd.Index(np.arange(0, max_week_test + 1), name="week")
surv_wide = surv_wide.reindex(full_week_index)
surv_wide = surv_wide.ffill()
surv_wide = surv_wide.fillna(1.0)
surv_df = surv_wide.copy()
surv_df.columns.name = "enrollment_id"

# ------------------------------
# 10) Primary survival metrics
# ------------------------------
eval_surv = EvalSurv(
    surv=surv_df,
    durations=durations_test,
    events=events_test,
    censor_surv="km",
)

primary_metrics_rows = []

try:
    c_index = float(eval_surv.concordance_td())
    c_index_note = "pycox_concordance_td"
except Exception as e:
    c_index = np.nan
    c_index_note = f"failed: {str(e)}"

primary_metrics_rows.append({
    "metric_name": "c_index",
    "metric_category": "co_primary",
    "metric_value": c_index,
    "notes": c_index_note,
})

try:
    max_requested_horizon = int(max(HORIZONS_WEEKS))
    ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
    ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    ibs_note = "pycox_integrated_brier_score"
except Exception as e:
    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        dense_brier = eval_surv.brier_score(ibs_grid)
        ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
        ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
    except Exception as e2:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

primary_metrics_rows.insert(0, {
    "metric_name": "ibs",
    "metric_category": "primary",
    "metric_value": ibs_value,
    "notes": ibs_note,
})

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 11) Brier by horizon from EvalSurv
# ------------------------------
try:
    brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": list(brier_h.index.astype(int)),
        "metric_name": ["brier_at_horizon"] * len(brier_h.index),
        "metric_category": ["primary"] * len(brier_h.index),
        "metric_value": list(brier_h.values.astype(float)),
        "notes": ["pycox_brier_score"] * len(brier_h.index),
    })
except Exception as e:
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": HORIZONS_WEEKS,
        "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
        "metric_category": ["primary"] * len(HORIZONS_WEEKS),
        "metric_value": [np.nan] * len(HORIZONS_WEEKS),
        "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
    })

# ------------------------------
# 12) Helpers for horizon extraction
# ------------------------------
def get_pred_at_horizon(df, h, value_col):
    eligible = df[df["week"] <= h].copy()
    if eligible.empty:
        return pd.DataFrame(columns=["enrollment_id", value_col])
    last_rows = (
        eligible.sort_values(["enrollment_id", "week"])
        .groupby("enrollment_id", as_index=False)
        .tail(1)
    )
    return last_rows[["enrollment_id", value_col]]

# ------------------------------
# 13) Support, calibration, risk AUC, predicted vs observed
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []
pred_vs_obs_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_pred_at_horizon(test_pred_df, h, "pred_survival").rename(columns={"pred_survival": "pred_survival_h"})
    pred_risk_h = get_pred_at_horizon(test_pred_df, h, "pred_risk").rename(columns={"pred_risk": "pred_risk_h"})

    eval_df = truth_test.merge(pred_surv_h, on="enrollment_id", how="left").merge(pred_risk_h, on="enrollment_id", how="left")

    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)

    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
    eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df.shape[0] > 0:
        ranked = eval_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))
        eval_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            eval_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)
calibration_bins_df = pd.DataFrame(calibration_rows)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_bins_df[calibration_bins_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan

    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 14) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []
if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            pred_risk_h = get_pred_at_horizon(test_pred_df, h, "pred_risk")
            merged = truth_test[["enrollment_id"]].merge(pred_risk_h, on="enrollment_id", how="left")
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                merged["pred_risk"].to_numpy(),
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)
secondary_metrics_df = pd.concat([risk_auc_by_horizon_df, time_dependent_auc_df], ignore_index=True)

# ------------------------------
# 15) Row-level diagnostics
# ------------------------------
if len(np.unique(y_test_linear)) >= 2:
    row_roc_auc = roc_auc_score(y_test_linear, test_pred_row)
    row_pr_auc = average_precision_score(y_test_linear, test_pred_row)
else:
    row_roc_auc = np.nan
    row_pr_auc = np.nan

row_log_loss = log_loss(y_test_linear, test_pred_row, labels=[0, 1])
row_brier = brier_score_loss(y_test_linear, test_pred_row)

row_diagnostics_df = pd.DataFrame([{
    "model_name": "linear_discrete_time_hazard_tuned",
    "row_level_roc_auc": float(row_roc_auc) if pd.notna(row_roc_auc) else np.nan,
    "row_level_pr_auc": float(row_pr_auc) if pd.notna(row_pr_auc) else np.nan,
    "row_level_log_loss": float(row_log_loss),
    "row_level_brier": float(row_brier),
}])

# ------------------------------
# 16) Save artifacts
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "linear_discrete_time_hazard_tuned.joblib"
preprocessor_path = model_dir / "linear_discrete_time_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_linear_tuning_results.csv"
predictions_path = TABLES_DIR / "table_linear_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_linear_tuned_primary_metrics.csv"
brier_path = TABLES_DIR / "table_linear_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_linear_tuned_secondary_metrics.csv"
row_diag_path = TABLES_DIR / "table_linear_tuned_row_diagnostics.csv"
support_path = TABLES_DIR / "table_linear_tuned_support_by_horizon.csv"
cal_summary_path = TABLES_DIR / "table_linear_tuned_calibration_summary.csv"
cal_bins_path = TABLES_DIR / "table_linear_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_linear_tuned_predicted_vs_observed_survival.csv"
config_path = METADATA_DIR / "linear_tuned_model_config.json"

joblib.dump(best_model, model_path)
if linear_preprocessor_obj is not None:
    joblib.dump(linear_preprocessor_obj, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
test_pred_df.to_csv(predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_by_horizon_df.to_csv(brier_path, index=False)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)
row_diagnostics_df.to_csv(row_diag_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(cal_summary_path, index=False)
calibration_bins_df.to_csv(cal_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)

config_to_save = dict(LINEAR_TUNING_CONFIG)
config_to_save["best_candidate"] = {
    "penalty": best_penalty,
    "C": best_C,
}
save_json(config_to_save, config_path)

# ------------------------------
# 17) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(best_candidate_df)

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_by_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nRow-level diagnostics:")
display(row_diagnostics_df)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nSaved artifacts:")
print("-", model_path.resolve())
if linear_preprocessor_obj is not None:
    print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", row_diag_path.resolve())
print("-", support_path.resolve())
print("-", cal_summary_path.resolve())
print("-", cal_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", config_path.resolve())

## D3 — Treatment for the Neural Discrete-Time Survival Model


In [ ]:
# ==============================================================
# D3 — Treatment for the Neural Discrete-Time Survival Model
# CORRECTED VERSION (robust to canonical-vs-materialized feature names)
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the neural
#   discrete-time survival model, with all learned transformations
#   fitted on training data only.
#
# Methodological note:
#   This step intentionally uses the same conceptual feature set
#   adopted for the linear discrete-time hazard baseline, so that
#   the comparison between the linear and neural formulations is
#   driven primarily by model class rather than by a different
#   feature engineering design.
#
#   The following learned operations are fit on training data only:
#     - numeric imputation
#     - categorical imputation
#     - one-hot encoding
#     - numeric standardization
#
#   The same fitted preprocessing object is then applied to the
#   test data without re-estimating any parameter, preventing leakage.
#
# Design note:
#   After the benchmark-architecture refactor, P10 started exposing
#   a canonical minimal dynamic-arm config (e.g., "total_clicks"),
#   while the materialized person-period tables still use the richer
#   notebook-native names (e.g., "total_clicks_week").
#   This cell therefore resolves canonical aliases to the actual
#   materialized columns instead of requiring exact name identity.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - TEMPORAL_FEATURES_DISCRETE
#
# Outputs:
#   - X_train_neural, X_test_neural
#   - y_train_neural, y_test_neural
#   - neural_preprocessor
#   - neural_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D3 — Treatment for the Neural Discrete-Time Survival Model")
print("=" * 70)
print("Methodological note: the neural model uses the same conceptual")
print("feature set as the linear baseline for comparability.")
print("All learned preprocessing transformations are fit on training data only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "TEMPORAL_FEATURES_DISCRETE"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df_neural = con.execute("SELECT * FROM pp_neural_hazard_ready_train").fetchdf()
test_df_neural = con.execute("SELECT * FROM pp_neural_hazard_ready_test").fetchdf()

if train_df_neural.empty:
    raise ValueError("pp_neural_hazard_ready_train is empty.")
if test_df_neural.empty:
    raise ValueError("pp_neural_hazard_ready_test is empty.")

# ------------------------------
# 3) Define target and operational feature columns
# ------------------------------
target_col = "event_t"

# Notebook-native materialized columns currently used in the
# discrete-time neural person-period tables.
categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "week",
    "total_clicks_week",
    "active_this_week",
    "n_vle_rows_week",
    "n_distinct_sites_week",
    "cum_clicks_until_t",
    "recency",
    "streak",
]

neural_feature_columns = categorical_features + numeric_features

# ------------------------------
# 4) Canonical-to-materialized alias resolution
# ------------------------------
feature_alias_map = {
    "total_clicks": "total_clicks_week",
}

expected_features_raw = list(STATIC_FEATURES) + list(TEMPORAL_FEATURES_DISCRETE)
expected_features_resolved = [
    feature_alias_map.get(col, col) for col in expected_features_raw
]

missing_from_defined = [
    c for c in expected_features_resolved
    if c not in neural_feature_columns
]

missing_in_train = [c for c in neural_feature_columns if c not in train_df_neural.columns]
missing_in_test = [c for c in neural_feature_columns if c not in test_df_neural.columns]

if missing_from_defined:
    raise ValueError(
        "Configured canonical features are not covered by the operational "
        f"neural treatment spec after alias resolution: {missing_from_defined}. "
        f"Raw expected features were: {expected_features_raw}"
    )

if missing_in_train or missing_in_test:
    raise ValueError(
        "Operational neural feature columns are missing from the materialized "
        "train/test tables.\n"
        f"Missing in train: {missing_in_train}\n"
        f"Missing in test : {missing_in_test}"
    )

if target_col not in train_df_neural.columns or target_col not in test_df_neural.columns:
    raise ValueError(
        f"Target column '{target_col}' is missing from train/test tables."
    )

# ------------------------------
# 5) Split X / y
# ------------------------------
X_train_raw_neural = train_df_neural[neural_feature_columns].copy()
X_test_raw_neural = test_df_neural[neural_feature_columns].copy()

y_train_neural = train_df_neural[target_col].astype(np.float32).to_numpy()
y_test_neural = test_df_neural[target_col].astype(np.float32).to_numpy()

# ------------------------------
# 6) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

neural_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 7) Fit on train only, transform train/test
# ------------------------------
X_train_neural = neural_preprocessor.fit_transform(X_train_raw_neural).astype(np.float32)
X_test_neural = neural_preprocessor.transform(X_test_raw_neural).astype(np.float32)

neural_feature_names_out = neural_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 8) Build preprocessing summaries
# ------------------------------
summary_rows = [{
    "model_family": "neural_discrete_time_survival",
    "train_rows": int(X_train_neural.shape[0]),
    "test_rows": int(X_test_neural.shape[0]),
    "n_input_features_raw": int(len(neural_feature_columns)),
    "n_numeric_features": int(len(numeric_features)),
    "n_categorical_features": int(len(categorical_features)),
    "n_output_features_after_preprocessing": int(len(neural_feature_names_out)),
    "train_event_rate": float(y_train_neural.mean()),
    "test_event_rate": float(y_test_neural.mean()),
}]

neural_preprocessing_summary_df = pd.DataFrame(summary_rows)

feature_manifest_df = pd.DataFrame({
    "feature_name_operational": neural_feature_columns,
    "feature_role": [
        "categorical" if c in categorical_features else "numeric"
        for c in neural_feature_columns
    ],
    "present_in_train": [c in train_df_neural.columns for c in neural_feature_columns],
    "present_in_test": [c in test_df_neural.columns for c in neural_feature_columns],
    "covered_by_canonical_config_after_alias": [
        c in expected_features_resolved for c in neural_feature_columns
    ],
    "canonical_source_name": [
        next(
            (raw for raw, resolved in zip(expected_features_raw, expected_features_resolved) if resolved == c),
            ""
        )
        for c in neural_feature_columns
    ],
})

canonical_alignment_df = pd.DataFrame({
    "canonical_feature_raw": expected_features_raw,
    "canonical_feature_resolved": expected_features_resolved,
    "covered_by_operational_treatment_spec": [
        c in neural_feature_columns for c in expected_features_resolved
    ],
})

output_feature_manifest_df = pd.DataFrame({
    "preprocessed_feature_name": neural_feature_names_out
})

# ------------------------------
# 9) Save outputs
# ------------------------------
summary_path = TABLES_DIR / "table_p18_neural_treatment_summary.csv"
feature_manifest_path = TABLES_DIR / "table_p18_neural_feature_manifest.csv"
canonical_alignment_path = TABLES_DIR / "table_p18_neural_canonical_alignment.csv"
output_manifest_path = TABLES_DIR / "table_p18_neural_output_feature_manifest.csv"
config_path = METADATA_DIR / "metadata_p18_neural_treatment.json"

neural_preprocessing_summary_df.to_csv(summary_path, index=False)
feature_manifest_df.to_csv(feature_manifest_path, index=False)
canonical_alignment_df.to_csv(canonical_alignment_path, index=False)
output_feature_manifest_df.to_csv(output_manifest_path, index=False)

save_json(
    {
        "step": "P18",
        "model_family": "neural_discrete_time_survival",
        "train_table": "pp_neural_hazard_ready_train",
        "test_table": "pp_neural_hazard_ready_test",
        "target_column": target_col,
        "categorical_features": categorical_features,
        "numeric_features": numeric_features,
        "operational_feature_columns": neural_feature_columns,
        "canonical_expected_features_raw": expected_features_raw,
        "canonical_expected_features_resolved": expected_features_resolved,
        "feature_alias_map": feature_alias_map,
        "n_train_rows": int(X_train_neural.shape[0]),
        "n_test_rows": int(X_test_neural.shape[0]),
        "n_output_features_after_preprocessing": int(len(neural_feature_names_out)),
        "train_event_rate": float(y_train_neural.mean()),
        "test_event_rate": float(y_test_neural.mean()),
        "methodological_note": (
            "All learned preprocessing operations were fit on training data only "
            "and then applied unchanged to test data."
        ),
        "design_note": (
            "Canonical config features were resolved to notebook-native materialized "
            "column names before validation."
        ),
    },
    config_path,
)

# ------------------------------
# 10) Console summary
# ------------------------------
print("\nOperational feature columns used by the neural model:")
for col in neural_feature_columns:
    print(" -", col)

print("\nCanonical config alignment:")
display(canonical_alignment_df)

print("\nNeural preprocessing summary:")
display(neural_preprocessing_summary_df)

print("\nSaved:")
print("-", summary_path.resolve())
print("-", feature_manifest_path.resolve())
print("-", canonical_alignment_path.resolve())
print("-", output_manifest_path.resolve())
print("-", config_path.resolve())

### D3.1 — Performance-Oriented Neural Tuning Benchmark


In [ ]:
# ==============================================================
# D3.1 — Performance-Oriented Neural Tuning Benchmark
# --------------------------------------------------------------
# Purpose:
#   Run a controlled neural tuning benchmark to obtain a stronger
#   performance-oriented discrete-time survival model, while keeping
#   the evaluation protocol aligned with the benchmark design.
#
# Methodological note:
#   This step materializes the official tuned neural benchmark.
#   It introduces a controlled tuning phase intended to improve performance without
#   turning the benchmark into an unbounded optimization exercise.
#
#   The search remains deliberately limited and reproducible.
#   Candidate models are compared under the same enrollment-level
#   internal validation strategy used to avoid leakage.
#
# Inputs expected from previous cells:
#   - con
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - X_train_neural, X_test_neural
#   - y_train_neural, y_test_neural
#   - train_df_neural, test_df_neural
#   - neural_preprocessor
#   - neural_feature_names_out
#   - HORIZONS_WEEKS
#   - CALIBRATION_BINS
#
# Outputs:
#   - tuning results table
#   - best tuned neural model
#   - test-time hazard/survival predictions
#   - primary / secondary / diagnostic metric tables
#   - calibration tables by horizon
#   - saved model artifacts and evaluation outputs
# ==============================================================

print("\n" + "=" * 70)
print("D3.1 — Performance-Oriented Neural Tuning Benchmark")
print("=" * 70)
print("Methodological note: this step performs controlled neural tuning.")
print("This step is the official stronger neural benchmark under the")
print("same evaluation protocol and tuned-model consolidation strategy.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_neural", "X_test_neural", "y_train_neural", "y_test_neural",
    "train_df_neural", "test_df_neural", "neural_preprocessor", "neural_feature_names_out",
    "HORIZONS_WEEKS", "CALIBRATION_BINS"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split

try:
    from pycox.evaluation import EvalSurv
    PYCOX_EVAL_AVAILABLE = True
except Exception:
    PYCOX_EVAL_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Global tuning config
# ------------------------------
TUNING_CONFIG = {
    "model_name": "neural_discrete_time_survival_tuned",
    "search_strategy": "controlled_grid_search",
    "selection_criterion": "lowest_validation_loss",
    "validation_split_unit": "enrollment",
    "validation_fraction": 0.10,
    "batch_size": 1024,
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "optimizer": "Adam",
    "loss": "BCEWithLogitsLoss",
    "random_state": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "benchmark_positioning_note": (
        "Performance-oriented neural benchmark following the simple benchmark "
        "version from P19, while preserving the same preprocessing and "
        "evaluation protocol."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

device = torch.device(TUNING_CONFIG["device"])

# ------------------------------
# 4) Internal validation split at enrollment level
# ------------------------------
train_enrollment_ids = train_df_neural["enrollment_id"].drop_duplicates().tolist()

enr_train_ids, enr_val_ids = train_test_split(
    train_enrollment_ids,
    test_size=TUNING_CONFIG["validation_fraction"],
    random_state=TUNING_CONFIG["random_state"],
)

enr_train_ids = set(enr_train_ids)
enr_val_ids = set(enr_val_ids)

internal_split = train_df_neural["enrollment_id"].map(
    lambda x: "train_internal" if x in enr_train_ids else "val_internal"
)

train_internal_mask = (internal_split == "train_internal").to_numpy()
val_internal_mask = (internal_split == "val_internal").to_numpy()

X_train_internal = X_train_neural[train_internal_mask]
y_train_internal = y_train_neural[train_internal_mask]

X_val_internal = X_train_neural[val_internal_mask]
y_val_internal = y_train_neural[val_internal_mask]

# torch tensors
X_train_tensor = torch.from_numpy(X_train_internal.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train_internal.astype(np.float32)).view(-1, 1)

X_val_tensor = torch.from_numpy(X_val_internal.astype(np.float32))
y_val_tensor = torch.from_numpy(y_val_internal.astype(np.float32)).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)


### Classe: TunedHazardMLP

Definicao isolada de TunedHazardMLP para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 5) Model class
# ------------------------------
class TunedHazardMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


In [ ]:

# ------------------------------
# 6) Tuning grid
# ------------------------------
grid_hidden_dims = [[64, 32], [128, 64]]
grid_dropout = [0.10, 0.30]
grid_learning_rate = [1e-3, 5e-4]
grid_weight_decay = [1e-5, 1e-4]

candidate_grid = list(itertools.product(
    grid_hidden_dims,
    grid_dropout,
    grid_learning_rate,
    grid_weight_decay
))

tuning_rows = []
best_candidate = None
best_val_loss = float("inf")
best_state_dict = None
best_model_obj = None

for candidate_id, (hidden_dims, dropout, lr, weight_decay) in enumerate(candidate_grid, start=1):
    torch.manual_seed(TUNING_CONFIG["random_state"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(TUNING_CONFIG["random_state"])

    model = TunedHazardMLP(
        input_dim=int(X_train_neural.shape[1]),
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=TUNING_CONFIG["batch_size"],
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=TUNING_CONFIG["batch_size"],
        shuffle=False,
        drop_last=False,
    )

    candidate_best_val = float("inf")
    candidate_best_epoch = None
    candidate_best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, TUNING_CONFIG["max_epochs"] + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())

        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))

        if mean_val_loss < candidate_best_val - 1e-6:
            candidate_best_val = mean_val_loss
            candidate_best_epoch = epoch
            candidate_best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= TUNING_CONFIG["early_stopping_patience"]:
            break

    tuning_rows.append({
        "candidate_id": candidate_id,
        "hidden_dims": str(hidden_dims),
        "dropout": dropout,
        "learning_rate": lr,
        "weight_decay": weight_decay,
        "best_val_loss": candidate_best_val,
        "best_epoch": candidate_best_epoch,
    })

    if candidate_best_val < best_val_loss:
        best_val_loss = candidate_best_val
        best_candidate = {
            "candidate_id": candidate_id,
            "hidden_dims": hidden_dims,
            "dropout": dropout,
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "best_val_loss": candidate_best_val,
            "best_epoch": candidate_best_epoch,
        }
        best_state_dict = candidate_best_state
        best_model_obj = TunedHazardMLP(
            input_dim=int(X_train_neural.shape[1]),
            hidden_dims=hidden_dims,
            dropout=dropout,
        ).to(device)

tuning_results_df = pd.DataFrame(tuning_rows).sort_values(
    ["best_val_loss", "candidate_id"], ascending=[True, True]
).reset_index(drop=True)

# ------------------------------
# 7) Final model = best candidate loaded with best state
# ------------------------------
best_model_obj.load_state_dict(best_state_dict)
best_model_obj.eval()


### Funcao: predict_proba_in_batches

Definicao isolada de predict_proba_in_batches para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 8) Prediction helper
# ------------------------------
def predict_proba_in_batches(model, X_np: np.ndarray, batch_size: int = 4096) -> np.ndarray:
    model.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, X_np.shape[0], batch_size):
            xb = torch.from_numpy(X_np[start:start+batch_size].astype(np.float32)).to(device)
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy().reshape(-1)
            probs.append(p)
    return np.concatenate(probs, axis=0)


In [ ]:

train_hazard_tuned = predict_proba_in_batches(best_model_obj, X_train_neural)
test_hazard_tuned = predict_proba_in_batches(best_model_obj, X_test_neural)

eps = 1e-8
train_hazard_tuned = np.clip(train_hazard_tuned, eps, 1 - eps)
test_hazard_tuned = np.clip(test_hazard_tuned, eps, 1 - eps)

# ------------------------------
# 9) Row-level diagnostic metrics
# ------------------------------
row_diagnostics = pd.DataFrame([{
    "model_name": "neural_discrete_time_survival_tuned",
    "row_level_roc_auc": roc_auc_score(y_test_neural, test_hazard_tuned),
    "row_level_pr_auc": average_precision_score(y_test_neural, test_hazard_tuned),
    "row_level_log_loss": log_loss(y_test_neural, test_hazard_tuned),
    "row_level_brier": brier_score_loss(y_test_neural, test_hazard_tuned),
}])

# ------------------------------
# 10) Reconstruct test hazard/survival trajectories
# ------------------------------
test_pred_df_tuned = test_df_neural.copy()
test_pred_df_tuned["pred_hazard"] = test_hazard_tuned
test_pred_df_tuned = test_pred_df_tuned.sort_values(["enrollment_id", "week"]).reset_index(drop=True)


### Funcao: compute_survival

Definicao isolada de compute_survival para reutilizacao nas etapas seguintes.


In [ ]:

def compute_survival(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy()
    g["pred_survival"] = (1.0 - g["pred_hazard"]).cumprod()
    g["pred_risk"] = 1.0 - g["pred_survival"]
    return g


In [ ]:

test_pred_df_tuned = (
    test_pred_df_tuned.groupby("enrollment_id", group_keys=False)
    .apply(compute_survival)
    .reset_index(drop=True)
)

# ------------------------------
# 11) Enrollment-level truth
# ------------------------------
truth_test = con.execute("""
SELECT
    enrollment_id,
    event,
    duration
FROM enrollment_cox_ready_test
ORDER BY enrollment_id
""").fetchdf()

truth_train = con.execute("""
SELECT
    enrollment_id,
    event,
    duration
FROM enrollment_cox_ready_train
ORDER BY enrollment_id
""").fetchdf()

truth_test["event"] = truth_test["event"].astype(int)
truth_test["duration"] = truth_test["duration"].astype(int)

truth_train["event"] = truth_train["event"].astype(int)
truth_train["duration"] = truth_train["duration"].astype(int)

# ------------------------------
# 12) Build survival/risk matrices
# ------------------------------
max_duration_test = int(truth_test["duration"].max())
eval_time_grid_full = np.arange(0, max_duration_test + 1, dtype=int)

surv_dict = {}
risk_dict = {}

for enrollment_id, group in test_pred_df_tuned.groupby("enrollment_id"):
    g = group.sort_values("week")

    surv_series = pd.Series(
        g["pred_survival"].values,
        index=g["week"].astype(int).values,
        name=enrollment_id,
    )
    risk_series = pd.Series(
        g["pred_risk"].values,
        index=g["week"].astype(int).values,
        name=enrollment_id,
    )

    surv_full = surv_series.reindex(eval_time_grid_full).ffill().fillna(1.0)
    risk_full = risk_series.reindex(eval_time_grid_full).ffill().fillna(0.0)

    surv_dict[enrollment_id] = surv_full.values
    risk_dict[enrollment_id] = risk_full.values

surv_df = pd.DataFrame(surv_dict, index=eval_time_grid_full)
risk_df = pd.DataFrame(risk_dict, index=eval_time_grid_full)

surv_df.columns.name = "enrollment_id"
risk_df.columns.name = "enrollment_id"

truth_test = truth_test.set_index("enrollment_id").loc[surv_df.columns].reset_index()
durations_test = truth_test["duration"].to_numpy(dtype=int)
events_test = truth_test["event"].to_numpy(dtype=int)


### Funcao: get_pred_risk_at_horizon

Definicao isolada de get_pred_risk_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 13) Helper functions
# ------------------------------
def get_pred_risk_at_horizon(h: int, risk_matrix: pd.DataFrame) -> pd.DataFrame:
    if h not in risk_matrix.index:
        raise ValueError(f"Horizon {h} not present in risk_df index.")
    return (
        risk_matrix.loc[h]
        .rename("pred_risk_h")
        .rename_axis("enrollment_id")
        .reset_index()
    )


### Funcao: get_pred_survival_at_horizon

Definicao isolada de get_pred_survival_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:

def get_pred_survival_at_horizon(h: int, surv_matrix: pd.DataFrame) -> pd.DataFrame:
    if h not in surv_matrix.index:
        raise ValueError(f"Horizon {h} not present in surv_df index.")
    return (
        surv_matrix.loc[h]
        .rename("pred_survival_h")
        .rename_axis("enrollment_id")
        .reset_index()
    )


In [ ]:

# ------------------------------
# 14) Primary survival metrics
# ------------------------------
primary_metrics_rows = []
brier_at_horizon_df = pd.DataFrame(columns=[
    "horizon_week", "metric_name", "metric_category", "metric_value", "notes"
])

if PYCOX_EVAL_AVAILABLE:
    eval_surv = EvalSurv(
        surv=surv_df,
        durations=durations_test,
        events=events_test,
        censor_surv="km",
    )

    try:
        c_index = float(eval_surv.concordance_td())
        c_index_note = "pycox_concordance_td"
    except Exception as e:
        c_index = np.nan
        c_index_note = f"failed: {str(e)}"

    primary_metrics_rows.append({
        "metric_name": "c_index",
        "metric_category": "co_primary",
        "metric_value": c_index,
        "notes": c_index_note,
    })

    try:
        brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
        brier_at_horizon_df = pd.DataFrame({
            "horizon_week": list(brier_h.index.astype(int)),
            "metric_name": ["brier_at_horizon"] * len(brier_h.index),
            "metric_category": ["primary"] * len(brier_h.index),
            "metric_value": list(brier_h.values.astype(float)),
            "notes": ["pycox_brier_score"] * len(brier_h.index),
        })
    except Exception as e:
        brier_at_horizon_df = pd.DataFrame({
            "horizon_week": HORIZONS_WEEKS,
            "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
            "metric_category": ["primary"] * len(HORIZONS_WEEKS),
            "metric_value": [np.nan] * len(HORIZONS_WEEKS),
            "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
        })

    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
        ibs_note = "pycox_integrated_brier_score"
    except Exception as e:
        try:
            max_requested_horizon = int(max(HORIZONS_WEEKS))
            ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
            dense_brier = eval_surv.brier_score(ibs_grid)
            ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
            ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
        except Exception as e2:
            ibs_value = np.nan
            ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

    primary_metrics_rows.insert(0, {
        "metric_name": "ibs",
        "metric_category": "primary",
        "metric_value": ibs_value,
        "notes": ibs_note,
    })
else:
    primary_metrics_rows.append({
        "metric_name": "ibs",
        "metric_category": "primary",
        "metric_value": np.nan,
        "notes": "pycox_unavailable",
    })
    primary_metrics_rows.append({
        "metric_name": "c_index",
        "metric_category": "co_primary",
        "metric_value": np.nan,
        "notes": "pycox_unavailable",
    })

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 15) Horizon-specific support, risk AUC, and calibration
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []

for h in HORIZONS_WEEKS:
    eval_df = truth_test.copy()
    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)

    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()

    pred_h = get_pred_risk_at_horizon(h, risk_df)
    eval_df = eval_df.merge(pred_h, on="enrollment_id", how="left")

    eval_df["observed_event_by_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    )

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    calib_df = eval_df[["enrollment_id", "pred_risk_h", "observed_event_by_h"]].copy()

    if calib_df.shape[0] > 0:
        ranked = calib_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, calib_df.shape[0])))

        calib_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            calib_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
calibration_by_horizon_df = pd.DataFrame(calibration_rows)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_by_horizon_df[calibration_by_horizon_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan

    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 16) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []

if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            risk_scores_h = risk_df.loc[h].to_numpy()
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                risk_scores_h,
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)

# ------------------------------
# 17) Predicted vs observed survival by horizon
# ------------------------------
pred_vs_obs_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_pred_survival_at_horizon(h, surv_df)
    tmp = truth_test.merge(pred_surv_h, on="enrollment_id", how="left")

    tmp["is_evaluable_at_h"] = (
        ((tmp["event"] == 1) & (tmp["duration"] <= h)) |
        (tmp["duration"] >= h)
    ).astype(int)
    tmp = tmp[tmp["is_evaluable_at_h"] == 1].copy()

    tmp["observed_survival_by_h"] = 1 - (
        ((tmp["event"] == 1) & (tmp["duration"] <= h)).astype(int)
    )

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(tmp.shape[0]),
        "mean_predicted_survival": float(tmp["pred_survival_h"].mean()) if tmp.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(tmp["observed_survival_by_h"].mean()) if tmp.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(tmp["pred_survival_h"].mean() - tmp["observed_survival_by_h"].mean())) if tmp.shape[0] > 0 else np.nan,
    })

predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)

# ------------------------------
# 18) Hazard and survival audit tables
# ------------------------------
hazard_audit_summary = pd.DataFrame([{
    "mean_pred_hazard_test": float(np.mean(test_hazard_tuned)),
    "median_pred_hazard_test": float(np.median(test_hazard_tuned)),
    "p90_pred_hazard_test": float(np.quantile(test_hazard_tuned, 0.90)),
    "p99_pred_hazard_test": float(np.quantile(test_hazard_tuned, 0.99)),
    "min_pred_hazard_test": float(np.min(test_hazard_tuned)),
    "max_pred_hazard_test": float(np.max(test_hazard_tuned)),
}])

hazard_by_week_df = (
    test_pred_df_tuned.groupby("week")
    .agg(
        mean_pred_hazard=("pred_hazard", "mean"),
        median_pred_hazard=("pred_hazard", "median"),
        mean_pred_survival=("pred_survival", "mean"),
        mean_pred_risk=("pred_risk", "mean"),
        n_rows=("enrollment_id", "count"),
    )
    .reset_index()
    .sort_values("week")
)

# ------------------------------
# 19) Save model and outputs
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "neural_discrete_time_survival_tuned.pt"
preprocessor_path = model_dir / "neural_discrete_time_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_neural_tuning_results.csv"
history_path = TABLES_DIR / "table_neural_tuned_training_history.csv"
test_predictions_path = TABLES_DIR / "table_neural_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_neural_tuned_primary_metrics.csv"
brier_horizon_path = TABLES_DIR / "table_neural_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_neural_tuned_secondary_metrics.csv"
diagnostics_path = TABLES_DIR / "table_neural_tuned_row_diagnostics.csv"
support_path = TABLES_DIR / "table_neural_tuned_support_by_horizon.csv"
calibration_summary_path = TABLES_DIR / "table_neural_tuned_calibration_summary.csv"
calibration_bins_path = TABLES_DIR / "table_neural_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_neural_tuned_predicted_vs_observed_survival.csv"
hazard_audit_path = TABLES_DIR / "table_neural_tuned_hazard_audit_summary.csv"
hazard_by_week_path = TABLES_DIR / "table_neural_tuned_hazard_by_week.csv"
model_config_path = METADATA_DIR / "neural_tuned_model_config.json"

torch.save(best_model_obj.state_dict(), model_path)
joblib.dump(neural_preprocessor, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
# final chosen candidate history is not stored per-candidate during search; store best candidate summary
pd.DataFrame([best_candidate]).to_csv(history_path, index=False)
test_pred_df_tuned.to_csv(test_predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_at_horizon_df.to_csv(brier_horizon_path, index=False)

secondary_metrics_df = pd.concat(
    [risk_auc_by_horizon_df, time_dependent_auc_df],
    ignore_index=True
)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)

row_diagnostics.to_csv(diagnostics_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(calibration_summary_path, index=False)
calibration_by_horizon_df.to_csv(calibration_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)
hazard_audit_summary.to_csv(hazard_audit_path, index=False)
hazard_by_week_df.to_csv(hazard_by_week_path, index=False)

best_config_to_save = dict(TUNING_CONFIG)
best_config_to_save["best_candidate"] = best_candidate
save_json(best_config_to_save, model_config_path)

# ------------------------------
# 20) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(pd.DataFrame([best_candidate]))

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_at_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nRow-level diagnostics:")
display(row_diagnostics)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nHazard audit summary:")
display(hazard_audit_summary)

print("\nHazard by week (first rows):")
display(hazard_by_week_df.head(20))

print("\nSaved artifacts:")
print("-", model_path.resolve())
print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", history_path.resolve())
print("-", test_predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_horizon_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", diagnostics_path.resolve())
print("-", support_path.resolve())
print("-", calibration_summary_path.resolve())
print("-", calibration_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", hazard_audit_path.resolve())
print("-", hazard_by_week_path.resolve())
print("-", model_config_path.resolve())


## D4 — Treatment for Cox (Revised)


In [ ]:
# ==============================================================
# D4 — Treatment for Cox (Revised)
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the revised
#   Cox benchmark, with all learned transformations fitted on
#   training data only.
#
# Methodological note:
#   This revised Cox treatment aligns with the early-window
#   comparable benchmark design.
#
#   Therefore, the Cox model now uses:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   This avoids retrospective whole-course summaries in the main
#   Cox benchmark track.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - MAIN_ENROLLMENT_WINDOW_FEATURES
#
# Outputs:
#   - X_train_cox, X_test_cox
#   - duration_train_cox, duration_test_cox
#   - event_train_cox, event_test_cox
#   - cox_preprocessor
#   - cox_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D4 — Treatment for Cox (Revised)")
print("=" * 70)
print("Methodological note: this Cox treatment now follows the")
print("early-window comparable benchmark design.")
print("All learned preprocessing transformations are fit on training data only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "MAIN_ENROLLMENT_WINDOW_FEATURES"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df_cox = con.execute("SELECT * FROM enrollment_cox_ready_train").fetchdf()
test_df_cox = con.execute("SELECT * FROM enrollment_cox_ready_test").fetchdf()

# ------------------------------
# 3) Define targets and feature columns
# ------------------------------
duration_col = "duration"
event_col = "event"

categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

cox_feature_columns = categorical_features + numeric_features

expected_features = STATIC_FEATURES + MAIN_ENROLLMENT_WINDOW_FEATURES
missing_from_defined = [c for c in expected_features if c not in cox_feature_columns]
if missing_from_defined:
    raise ValueError(f"Missing configured Cox features in treatment spec: {missing_from_defined}")

# ------------------------------
# 4) Split X / targets
# ------------------------------
X_train_raw_cox = train_df_cox[cox_feature_columns].copy()
X_test_raw_cox = test_df_cox[cox_feature_columns].copy()

duration_train_cox = train_df_cox[duration_col].astype(np.float32).to_numpy()
duration_test_cox = test_df_cox[duration_col].astype(np.float32).to_numpy()

event_train_cox = train_df_cox[event_col].astype(np.int32).to_numpy()
event_test_cox = test_df_cox[event_col].astype(np.int32).to_numpy()

# ------------------------------
# 5) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

cox_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 6) Fit on train only, transform train/test
# ------------------------------
X_train_cox = cox_preprocessor.fit_transform(X_train_raw_cox).astype(np.float32)
X_test_cox = cox_preprocessor.transform(X_test_raw_cox).astype(np.float32)

cox_feature_names_out = cox_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 7) Build preprocessing summaries
# ------------------------------
summary_rows = [{
    "model_family": "cox",
    "train_rows": int(X_train_cox.shape[0]),
    "test_rows": int(X_test_cox.shape[0]),
    "n_input_features_raw": int(len(cox_feature_columns)),
    "n_numeric_features": int(len(numeric_features)),
    "n_categorical_features": int(len(categorical_features)),
    "n_features_after_transform": int(X_train_cox.shape[1]),
    "n_events_train": int(event_train_cox.sum()),
    "n_events_test": int(event_test_cox.sum()),
    "mean_duration_train": float(np.mean(duration_train_cox)),
    "mean_duration_test": float(np.mean(duration_test_cox)),
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "cox_positioning_note": "early-window comparable Cox benchmark",
    "window_weeks": 4,
}]

cox_preprocessing_summary = pd.DataFrame(summary_rows)
cox_feature_names_df = pd.DataFrame({
    "feature_name_out": cox_feature_names_out
})

# ------------------------------
# 8) Save metadata and audit tables
# ------------------------------
summary_path = TABLES_DIR / "table_cox_preprocessing_summary.csv"
feature_names_path = TABLES_DIR / "table_cox_feature_names_out.csv"
config_path = METADATA_DIR / "cox_preprocessing_config.json"

cox_preprocessing_summary.to_csv(summary_path, index=False)
cox_feature_names_df.to_csv(feature_names_path, index=False)

cox_preprocessing_config = {
    "model_family": "cox",
    "duration_column": duration_col,
    "event_column": event_col,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "raw_feature_columns": cox_feature_columns,
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "cox_positioning_note": "early-window comparable Cox benchmark",
    "window_weeks": 4,
    "n_features_after_transform": int(X_train_cox.shape[1]),
}

save_json(cox_preprocessing_config, config_path)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nCox preprocessing summary:")
display(cox_preprocessing_summary)

print("\nFirst transformed feature names:")
display(cox_feature_names_df.head(40))

print("\nArray shapes:")
print("X_train_cox     :", X_train_cox.shape, X_train_cox.dtype)
print("X_test_cox      :", X_test_cox.shape, X_test_cox.dtype)
print("duration_train  :", duration_train_cox.shape, duration_train_cox.dtype)
print("duration_test   :", duration_test_cox.shape, duration_test_cox.dtype)
print("event_train     :", event_train_cox.shape, event_train_cox.dtype)
print("event_test      :", event_test_cox.shape, event_test_cox.dtype)

print("\nSaved:")
print("-", summary_path.resolve())
print("-", feature_names_path.resolve())
print("-", config_path.resolve())

### D4.1 — Performance-Oriented Cox Tuning


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D4.1 — Performance-Oriented Cox Tuning
# --------------------------------------------------------------
# Purpose:
#   Perform a controlled hyperparameter search for the comparable
#   Cox benchmark, select the best candidate, refit it on the full
#   training set, and evaluate it under the same survival-oriented
#   benchmark protocol defined in D1.
#
# Methodological note:
#   This step provides the official tuned Cox benchmark under the same early-window
#   comparable design:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   The search is intentionally limited and reproducible.
# ==============================================================

print("\n" + "=" * 70)
print("D4.1 — Performance-Oriented Cox Tuning")
print("=" * 70)
print("Methodological note: this step performs controlled tuning for the")
print("early-window comparable Cox benchmark under the same survival-oriented")
print("benchmark protocol defined in D1.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_cox", "X_test_cox",
    "duration_train_cox", "duration_test_cox",
    "event_train_cox", "event_test_cox",
    "train_df_cox", "test_df_cox",
    "cox_preprocessor", "cox_feature_names_out",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
from sklearn.model_selection import train_test_split

try:
    from lifelines import CoxPHFitter
    LIFELINES_AVAILABLE = True
except Exception:
    LIFELINES_AVAILABLE = False

try:
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

if not LIFELINES_AVAILABLE:
    raise ImportError("lifelines is required for P22.2.")

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P22.2.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Build train/test frames
# ------------------------------
X_train_cox_df_full = pd.DataFrame(X_train_cox, columns=cox_feature_names_out)
X_test_cox_df_full = pd.DataFrame(X_test_cox, columns=cox_feature_names_out)

# zero-variance filter based on full train
train_std_full = X_train_cox_df_full.std(axis=0, ddof=0)
keep_cols = train_std_full[train_std_full > 0].index.tolist()
dropped_zero_var_cols = train_std_full[train_std_full <= 0].index.tolist()

X_train_cox_df_full = X_train_cox_df_full[keep_cols].copy()
X_test_cox_df_full = X_test_cox_df_full[keep_cols].copy()

train_cox_full_df = X_train_cox_df_full.copy()
train_cox_full_df["duration"] = duration_train_cox
train_cox_full_df["event"] = event_train_cox
train_cox_full_df["enrollment_id"] = train_df_cox["enrollment_id"].values

test_cox_eval_df = X_test_cox_df_full.copy()
test_cox_eval_df["duration"] = duration_test_cox
test_cox_eval_df["event"] = event_test_cox
test_cox_eval_df["enrollment_id"] = test_df_cox["enrollment_id"].values

# ------------------------------
# 4) Enrollment-level train/validation split
# ------------------------------
unique_train_enrollments = train_cox_full_df["enrollment_id"].drop_duplicates().to_numpy()

# stratify by event if possible
enrollment_event_df = (
    train_cox_full_df[["enrollment_id", "event"]]
    .drop_duplicates(subset=["enrollment_id"])
    .copy()
)

stratify_labels = enrollment_event_df["event"] if enrollment_event_df["event"].nunique() > 1 else None

train_ids, val_ids = train_test_split(
    enrollment_event_df["enrollment_id"].to_numpy(),
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=stratify_labels
)

subtrain_df = train_cox_full_df[train_cox_full_df["enrollment_id"].isin(train_ids)].copy()
val_df = train_cox_full_df[train_cox_full_df["enrollment_id"].isin(val_ids)].copy()

# Remove enrollment_id from fit frames
subtrain_fit_df = subtrain_df.drop(columns=["enrollment_id"]).copy()
val_fit_df = val_df.drop(columns=["enrollment_id"]).copy()

# ------------------------------
# 5) Tuning config
# ------------------------------
COX_TUNING_CONFIG = {
    "model_name": "cox_early_window_tuned",
    "penalizer_grid": [0.001, 0.01, 0.1, 1.0],
    "l1_ratio_grid": [0.0, 0.5, 1.0],
    "selection_metric": "val_neg_partial_log_likelihood",
    "window_weeks": 4,
    "benchmark_positioning_note": (
        "Performance-oriented tuned Cox benchmark under the same early-window comparable input design."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

# ------------------------------
# 6) Run tuning grid
# ------------------------------
candidate_rows = []

for candidate_id, (penalizer, l1_ratio) in enumerate(
    itertools.product(COX_TUNING_CONFIG["penalizer_grid"], COX_TUNING_CONFIG["l1_ratio_grid"]),
    start=1
):
    try:
        model = CoxPHFitter(
            penalizer=penalizer,
            l1_ratio=l1_ratio,
        )

        model.fit(
            subtrain_fit_df,
            duration_col="duration",
            event_col="event",
            show_progress=False,
        )

        # Lifelines reports log_likelihood_ on fitted data; use val score via score()
        val_score = model.score(val_fit_df, scoring_method="log_likelihood")
        val_c_index = model.score(val_fit_df, scoring_method="concordance_index")

        candidate_rows.append({
            "candidate_id": candidate_id,
            "penalizer": float(penalizer),
            "l1_ratio": float(l1_ratio),
            "val_neg_partial_log_likelihood": float(-val_score),
            "val_concordance_index": float(val_c_index),
            "fit_status": "ok",
        })
    except Exception as e:
        candidate_rows.append({
            "candidate_id": candidate_id,
            "penalizer": float(penalizer),
            "l1_ratio": float(l1_ratio),
            "val_neg_partial_log_likelihood": np.nan,
            "val_concordance_index": np.nan,
            "fit_status": f"failed: {str(e)}",
        })

tuning_results_df = pd.DataFrame(candidate_rows)

valid_tuning_df = tuning_results_df[tuning_results_df["fit_status"] == "ok"].copy()
if valid_tuning_df.empty:
    raise RuntimeError("All Cox tuning candidates failed. Review convergence and preprocessing.")

tuning_results_df = tuning_results_df.sort_values(
    by=["fit_status", "val_neg_partial_log_likelihood", "candidate_id"],
    ascending=[True, True, True]
).reset_index(drop=True)

best_valid_row = valid_tuning_df.sort_values(
    by=["val_neg_partial_log_likelihood", "candidate_id"],
    ascending=[True, True]
).iloc[0]

best_candidate_df = pd.DataFrame([best_valid_row.to_dict()])

best_penalizer = float(best_valid_row["penalizer"])
best_l1_ratio = float(best_valid_row["l1_ratio"])

# ------------------------------
# 7) Refit best model on full train
# ------------------------------
train_fit_df_full = train_cox_full_df.drop(columns=["enrollment_id"]).copy()

best_cox_model = CoxPHFitter(
    penalizer=best_penalizer,
    l1_ratio=best_l1_ratio,
)

best_cox_model.fit(
    train_fit_df_full,
    duration_col="duration",
    event_col="event",
    show_progress=False,
)

# ------------------------------
# 8) Predict survival on test
# ------------------------------
pred_surv = best_cox_model.predict_survival_function(
    X_test_cox_df_full,
    times=np.arange(0, int(np.max(duration_test_cox)) + 1, dtype=int)
)

surv_df = pred_surv.copy()
surv_df.columns = test_df_cox["enrollment_id"].tolist()
surv_df.columns.name = "enrollment_id"
surv_df.index = surv_df.index.astype(int)

truth_test = pd.DataFrame({
    "enrollment_id": test_df_cox["enrollment_id"].values,
    "event": event_test_cox.astype(int),
    "duration": duration_test_cox.astype(int),
})

truth_train = pd.DataFrame({
    "enrollment_id": train_df_cox["enrollment_id"].values,
    "event": event_train_cox.astype(int),
    "duration": duration_train_cox.astype(int),
})

durations_test = truth_test["duration"].to_numpy(dtype=int)
events_test = truth_test["event"].to_numpy(dtype=int)

# ------------------------------
# 9) Primary survival metrics
# ------------------------------
eval_surv = EvalSurv(
    surv=surv_df,
    durations=durations_test,
    events=events_test,
    censor_surv="km",
)

primary_metrics_rows = []

try:
    c_index = float(eval_surv.concordance_td())
    c_index_note = "pycox_concordance_td"
except Exception as e:
    c_index = np.nan
    c_index_note = f"failed: {str(e)}"

primary_metrics_rows.append({
    "metric_name": "c_index",
    "metric_category": "co_primary",
    "metric_value": c_index,
    "notes": c_index_note,
})

try:
    max_requested_horizon = int(max(HORIZONS_WEEKS))
    ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
    ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    ibs_note = "pycox_integrated_brier_score"
except Exception as e:
    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        dense_brier = eval_surv.brier_score(ibs_grid)
        ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
        ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
    except Exception as e2:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

primary_metrics_rows.insert(0, {
    "metric_name": "ibs",
    "metric_category": "primary",
    "metric_value": ibs_value,
    "notes": ibs_note,
})

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 10) Brier by horizon
# ------------------------------
try:
    brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": list(brier_h.index.astype(int)),
        "metric_name": ["brier_at_horizon"] * len(brier_h.index),
        "metric_category": ["primary"] * len(brier_h.index),
        "metric_value": list(brier_h.values.astype(float)),
        "notes": ["pycox_brier_score"] * len(brier_h.index),
    })
except Exception as e:
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": HORIZONS_WEEKS,
        "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
        "metric_category": ["primary"] * len(HORIZONS_WEEKS),
        "metric_value": [np.nan] * len(HORIZONS_WEEKS),
        "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
    })

# ------------------------------
# 11) Helpers
# ------------------------------
def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]

# ------------------------------
# 12) Support, calibration, risk AUC, predicted vs observed
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []
pred_vs_obs_rows = []
test_pred_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_surv_at_horizon(surv_df, h)
    pred_risk_h = 1.0 - pred_surv_h

    eval_df = truth_test.copy()
    eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
    eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)
    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()

    eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df.shape[0] > 0:
        ranked = eval_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))

        eval_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            eval_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

    tmp_preds = truth_test[["enrollment_id", "event", "duration"]].copy()
    tmp_preds["horizon_week"] = h
    tmp_preds["pred_survival_h"] = tmp_preds["enrollment_id"].map(pred_surv_h.to_dict())
    tmp_preds["pred_risk_h"] = tmp_preds["enrollment_id"].map(pred_risk_h.to_dict())
    test_pred_rows.append(tmp_preds)

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)
calibration_bins_df = pd.DataFrame(calibration_rows)
test_predictions_df = pd.concat(test_pred_rows, ignore_index=True)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_bins_df[calibration_bins_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan
    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 13) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []
if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            pred_risk_h = 1.0 - get_surv_at_horizon(surv_df, h)
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                pred_risk_h.to_numpy(),
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)
secondary_metrics_df = pd.concat([risk_auc_by_horizon_df, time_dependent_auc_df], ignore_index=True)

# ------------------------------
# 14) Save coefficient/stability tables
# ------------------------------
cox_coefficients_df = best_cox_model.summary.reset_index().rename(columns={"covariate": "feature_name"})

stability_notes_df = pd.DataFrame([{
    "n_input_features_before_filter": len(cox_feature_names_out),
    "n_input_features_after_filter": len(keep_cols),
    "n_zero_variance_features_dropped": len(dropped_zero_var_cols),
    "dropped_zero_variance_features": "; ".join(dropped_zero_var_cols) if dropped_zero_var_cols else "",
    "best_penalizer": best_penalizer,
    "best_l1_ratio": best_l1_ratio,
    "window_weeks": COX_TUNING_CONFIG["window_weeks"],
}])

# ------------------------------
# 15) Save artifacts
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "cox_early_window_tuned.joblib"
preprocessor_path = model_dir / "cox_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_cox_tuning_results.csv"
predictions_path = TABLES_DIR / "table_cox_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_cox_tuned_primary_metrics.csv"
brier_path = TABLES_DIR / "table_cox_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_cox_tuned_secondary_metrics.csv"
support_path = TABLES_DIR / "table_cox_tuned_support_by_horizon.csv"
cal_summary_path = TABLES_DIR / "table_cox_tuned_calibration_summary.csv"
cal_bins_path = TABLES_DIR / "table_cox_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_cox_tuned_predicted_vs_observed_survival.csv"
coef_path = TABLES_DIR / "table_cox_tuned_coefficients.csv"
stability_path = TABLES_DIR / "table_cox_tuned_stability_notes.csv"
config_path = METADATA_DIR / "cox_tuned_model_config.json"

joblib.dump(best_cox_model, model_path)
joblib.dump(cox_preprocessor, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
test_predictions_df.to_csv(predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_by_horizon_df.to_csv(brier_path, index=False)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(cal_summary_path, index=False)
calibration_bins_df.to_csv(cal_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)
cox_coefficients_df.to_csv(coef_path, index=False)
stability_notes_df.to_csv(stability_path, index=False)

config_to_save = dict(COX_TUNING_CONFIG)
config_to_save["best_candidate"] = {
    "penalizer": best_penalizer,
    "l1_ratio": best_l1_ratio,
}
save_json(config_to_save, config_path)

# ------------------------------
# 16) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(best_candidate_df)

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_by_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nTop Cox coefficients:")
display(cox_coefficients_df.head(20))

print("\nStability notes:")
display(stability_notes_df)

print("\nSaved artifacts:")
print("-", model_path.resolve())
print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", support_path.resolve())
print("-", cal_summary_path.resolve())
print("-", cal_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", coef_path.resolve())
print("-", stability_path.resolve())
print("-", config_path.resolve())

## D5 — Treatment for DeepSurv


In [ ]:
# ==============================================================
# D5 — Treatment for DeepSurv
# --------------------------------------------------------------
# Purpose:
#   Prepare the train/test preprocessing pipeline for the revised
#   DeepSurv benchmark, with all learned transformations fitted on
#   training data only.
#
# Methodological note:
#   This DeepSurv treatment aligns with the same early-window
#   comparable benchmark design used by the Cox model.
#
#   Therefore, the DeepSurv model uses:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   This preserves comparability between Cox and DeepSurv:
#   both models receive the same conceptual input information,
#   differing only in modeling strategy.
#
# Inputs expected from previous cells:
#   - con
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - STATIC_FEATURES
#   - MAIN_ENROLLMENT_WINDOW_FEATURES
#
# Outputs:
#   - X_train_deepsurv, X_test_deepsurv
#   - duration_train_deepsurv, duration_test_deepsurv
#   - event_train_deepsurv, event_test_deepsurv
#   - deepsurv_preprocessor
#   - deepsurv_feature_names_out
#   - audit tables / metadata
# ==============================================================

print("\n" + "=" * 70)
print("D5 — Treatment for DeepSurv")
print("=" * 70)
print("Methodological note: this DeepSurv treatment follows the")
print("same early-window comparable benchmark design used by Cox.")
print("All learned preprocessing transformations are fit on training data only.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "con", "TABLES_DIR", "METADATA_DIR", "save_json",
    "STATIC_FEATURES", "MAIN_ENROLLMENT_WINDOW_FEATURES"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ------------------------------
# 2) Load train/test tables
# ------------------------------
train_df_deepsurv = con.execute("SELECT * FROM enrollment_deepsurv_ready_train").fetchdf()
test_df_deepsurv = con.execute("SELECT * FROM enrollment_deepsurv_ready_test").fetchdf()

# ------------------------------
# 3) Define targets and feature columns
# ------------------------------
duration_col = "duration"
event_col = "event"

categorical_features = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "disability",
]

numeric_features = [
    "num_of_prev_attempts",
    "studied_credits",
    "clicks_first_4_weeks",
    "active_weeks_first_4",
    "mean_clicks_first_4_weeks",
]

deepsurv_feature_columns = categorical_features + numeric_features

expected_features = STATIC_FEATURES + MAIN_ENROLLMENT_WINDOW_FEATURES
missing_from_defined = [c for c in expected_features if c not in deepsurv_feature_columns]
if missing_from_defined:
    raise ValueError(f"Missing configured DeepSurv features in treatment spec: {missing_from_defined}")

# ------------------------------
# 4) Split X / targets
# ------------------------------
X_train_raw_deepsurv = train_df_deepsurv[deepsurv_feature_columns].copy()
X_test_raw_deepsurv = test_df_deepsurv[deepsurv_feature_columns].copy()

duration_train_deepsurv = train_df_deepsurv[duration_col].astype(np.float32).to_numpy()
duration_test_deepsurv = test_df_deepsurv[duration_col].astype(np.float32).to_numpy()

event_train_deepsurv = train_df_deepsurv[event_col].astype(np.int32).to_numpy()
event_test_deepsurv = test_df_deepsurv[event_col].astype(np.int32).to_numpy()

# ------------------------------
# 5) Build preprocessing pipelines
# ------------------------------
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

deepsurv_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# ------------------------------
# 6) Fit on train only, transform train/test
# ------------------------------
X_train_deepsurv = deepsurv_preprocessor.fit_transform(X_train_raw_deepsurv).astype(np.float32)
X_test_deepsurv = deepsurv_preprocessor.transform(X_test_raw_deepsurv).astype(np.float32)

deepsurv_feature_names_out = deepsurv_preprocessor.get_feature_names_out().tolist()

# ------------------------------
# 7) Build preprocessing summaries
# ------------------------------
summary_rows = [{
    "model_family": "deepsurv",
    "train_rows": int(X_train_deepsurv.shape[0]),
    "test_rows": int(X_test_deepsurv.shape[0]),
    "n_input_features_raw": int(len(deepsurv_feature_columns)),
    "n_numeric_features": int(len(numeric_features)),
    "n_categorical_features": int(len(categorical_features)),
    "n_features_after_transform": int(X_train_deepsurv.shape[1]),
    "n_events_train": int(event_train_deepsurv.sum()),
    "n_events_test": int(event_test_deepsurv.sum()),
    "mean_duration_train": float(np.mean(duration_train_deepsurv)),
    "mean_duration_test": float(np.mean(duration_test_deepsurv)),
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "comparability_note": "same conceptual input design as Cox benchmark",
    "window_weeks": 4,
}]

deepsurv_preprocessing_summary = pd.DataFrame(summary_rows)
deepsurv_feature_names_df = pd.DataFrame({
    "feature_name_out": deepsurv_feature_names_out
})

# ------------------------------
# 8) Save metadata and audit tables
# ------------------------------
summary_path = TABLES_DIR / "table_deepsurv_preprocessing_summary.csv"
feature_names_path = TABLES_DIR / "table_deepsurv_feature_names_out.csv"
config_path = METADATA_DIR / "deepsurv_preprocessing_config.json"

deepsurv_preprocessing_summary.to_csv(summary_path, index=False)
deepsurv_feature_names_df.to_csv(feature_names_path, index=False)

deepsurv_preprocessing_config = {
    "model_family": "deepsurv",
    "duration_column": duration_col,
    "event_column": event_col,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "raw_feature_columns": deepsurv_feature_columns,
    "numeric_imputation": "median",
    "categorical_imputation": "constant_missing",
    "categorical_encoding": "one_hot_handle_unknown_ignore",
    "numeric_scaling": "standard_scaler_fit_on_train_only",
    "output_dtype": "float32",
    "comparability_note": "same conceptual input design as Cox benchmark",
    "window_weeks": 4,
    "n_features_after_transform": int(X_train_deepsurv.shape[1]),
}

save_json(deepsurv_preprocessing_config, config_path)

# ------------------------------
# 9) Output for feedback
# ------------------------------
print("\nDeepSurv preprocessing summary:")
display(deepsurv_preprocessing_summary)

print("\nFirst transformed feature names:")
display(deepsurv_feature_names_df.head(40))

print("\nArray shapes:")
print("X_train_deepsurv     :", X_train_deepsurv.shape, X_train_deepsurv.dtype)
print("X_test_deepsurv      :", X_test_deepsurv.shape, X_test_deepsurv.dtype)
print("duration_train       :", duration_train_deepsurv.shape, duration_train_deepsurv.dtype)
print("duration_test        :", duration_test_deepsurv.shape, duration_test_deepsurv.dtype)
print("event_train          :", event_train_deepsurv.shape, event_train_deepsurv.dtype)
print("event_test           :", event_test_deepsurv.shape, event_test_deepsurv.dtype)

print("\nSaved:")
print("-", summary_path.resolve())
print("-", feature_names_path.resolve())
print("-", config_path.resolve())

### D5.1 — Performance-Oriented DeepSurv Tuning


### Funcao: get_surv_at_horizon

Definicao isolada de get_surv_at_horizon para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D5.1 — Performance-Oriented DeepSurv Tuning
# --------------------------------------------------------------
# Purpose:
#   Perform a controlled hyperparameter search for DeepSurv and
#   evaluate the best candidate under the benchmark protocol.
#
# Methodological note:
#   This step provides the official stronger performance-oriented DeepSurv benchmark
#   under the same information constraints:
#     - static covariates
#     - early-window temporal summaries (main window = 4 weeks)
#
#   The search is intentionally limited and reproducible.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - TABLES_DIR
#   - METADATA_DIR
#   - save_json
#   - X_train_deepsurv, X_test_deepsurv
#   - duration_train_deepsurv, duration_test_deepsurv
#   - event_train_deepsurv, event_test_deepsurv
#   - train_df_deepsurv, test_df_deepsurv
#   - deepsurv_preprocessor
#   - deepsurv_feature_names_out
#   - HORIZONS_WEEKS
#   - CALIBRATION_BINS
#   - RANDOM_SEED
#
# Outputs:
#   - tuned DeepSurv model
#   - tuning results
#   - training history for best model
#   - test predictions
#   - benchmark metrics and diagnostics
# ==============================================================

print("\n" + "=" * 70)
print("D5.1 — Performance-Oriented DeepSurv Tuning")
print("=" * 70)
print("Methodological note: this step performs controlled DeepSurv tuning.")
print("This step is the official stronger DeepSurv benchmark under the")
print("performance-oriented DeepSurv benchmark under the same evaluation protocol.")

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = [
    "OUTPUT_DIR", "TABLES_DIR", "METADATA_DIR", "save_json",
    "X_train_deepsurv", "X_test_deepsurv",
    "duration_train_deepsurv", "duration_test_deepsurv",
    "event_train_deepsurv", "event_test_deepsurv",
    "train_df_deepsurv", "test_df_deepsurv",
    "deepsurv_preprocessor", "deepsurv_feature_names_out",
    "HORIZONS_WEEKS", "CALIBRATION_BINS", "RANDOM_SEED"
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please rerun the required prior steps first."
    )

import copy
import itertools
import joblib
import numpy as np
import pandas as pd
import scipy
import torch
import torchtuples as tt
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

try:
    from pycox.models import CoxPH
    from pycox.evaluation import EvalSurv
    PYCOX_AVAILABLE = True
except Exception:
    PYCOX_AVAILABLE = False

try:
    from sksurv.metrics import cumulative_dynamic_auc
    from sksurv.util import Surv
    SKSURV_AVAILABLE = True
except Exception:
    SKSURV_AVAILABLE = False

if not PYCOX_AVAILABLE:
    raise ImportError("pycox is required for P25.")

# ------------------------------
# 2) Compatibility patch for SciPy / PyCox
# ------------------------------
SCIPY_SIMPS_PATCHED = False
SCIPY_SIMPS_NOTE = "not_needed"

try:
    if not hasattr(scipy.integrate, "simps") and hasattr(scipy.integrate, "simpson"):
        def _simps_compat(y, x=None, dx=1.0, axis=-1, even=None):
            return scipy.integrate.simpson(y, x=x, dx=dx, axis=axis)
        scipy.integrate.simps = _simps_compat
        SCIPY_SIMPS_PATCHED = True
        SCIPY_SIMPS_NOTE = "patched_simps_to_simpson"
except Exception as e:
    SCIPY_SIMPS_NOTE = f"patch_failed: {str(e)}"

# ------------------------------
# 3) Reproducibility
# ------------------------------
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ------------------------------
# 4) Tuning config
# ------------------------------
DEEPSURV_TUNING_CONFIG = {
    "model_name": "deepsurv_tuned",
    "window_weeks": 4,
    "hidden_dims_grid": [[32, 16], [64, 32], [128, 64]],
    "dropout_grid": [0.1, 0.3],
    "learning_rate_grid": [5e-4, 1e-3],
    "weight_decay_grid": [1e-5, 1e-4],
    "batch_norm": True,
    "output_bias": False,
    "batch_size": 256,
    "epochs": 100,
    "patience": 10,
    "validation_fraction": 0.20,
    "selection_metric": "best_val_loss",
    "benchmark_positioning_note": (
        "Performance-oriented tuned DeepSurv benchmark under the same early-window comparable input design."
    ),
    "scipy_simps_note": SCIPY_SIMPS_NOTE,
}

# ------------------------------
# 5) Train/validation split at enrollment level
# ------------------------------
n_train = X_train_deepsurv.shape[0]
perm = np.random.RandomState(RANDOM_SEED).permutation(n_train)
val_size = int(np.floor(DEEPSURV_TUNING_CONFIG["validation_fraction"] * n_train))
val_idx = perm[:val_size]
subtrain_idx = perm[val_size:]

X_subtrain = X_train_deepsurv[subtrain_idx]
X_val = X_train_deepsurv[val_idx]

dur_subtrain = duration_train_deepsurv[subtrain_idx]
dur_val = duration_train_deepsurv[val_idx]

evt_subtrain = event_train_deepsurv[subtrain_idx]
evt_val = event_train_deepsurv[val_idx]

y_subtrain = (dur_subtrain.astype(np.float32), evt_subtrain.astype(np.float32))
y_val = (dur_val.astype(np.float32), evt_val.astype(np.float32))
y_train_full = (
    duration_train_deepsurv.astype(np.float32),
    event_train_deepsurv.astype(np.float32),
)

# ------------------------------
# 6) Run tuning grid
# ------------------------------
candidate_rows = []
candidate_artifacts = []

param_grid = list(itertools.product(
    DEEPSURV_TUNING_CONFIG["hidden_dims_grid"],
    DEEPSURV_TUNING_CONFIG["dropout_grid"],
    DEEPSURV_TUNING_CONFIG["learning_rate_grid"],
    DEEPSURV_TUNING_CONFIG["weight_decay_grid"],
))

in_features = X_train_deepsurv.shape[1]

for candidate_id, (hidden_dims, dropout, lr, wd) in enumerate(param_grid, start=1):
    torch.manual_seed(RANDOM_SEED + candidate_id)
    np.random.seed(RANDOM_SEED + candidate_id)

    net = tt.practical.MLPVanilla(
        in_features=in_features,
        num_nodes=hidden_dims,
        out_features=1,
        batch_norm=DEEPSURV_TUNING_CONFIG["batch_norm"],
        dropout=dropout,
        output_bias=DEEPSURV_TUNING_CONFIG["output_bias"],
    )

    model = CoxPH(net, tt.optim.AdamW)
    model.optimizer.set_lr(lr)

    callbacks = [tt.callbacks.EarlyStopping(patience=DEEPSURV_TUNING_CONFIG["patience"])]

    log = model.fit(
        X_subtrain,
        y_subtrain,
        batch_size=DEEPSURV_TUNING_CONFIG["batch_size"],
        epochs=DEEPSURV_TUNING_CONFIG["epochs"],
        callbacks=callbacks,
        verbose=False,
        val_data=(X_val, y_val),
    )

    history_df = log.to_pandas().reset_index().rename(columns={"index": "epoch"})
    if "train_loss" not in history_df.columns and "loss" in history_df.columns:
        history_df = history_df.rename(columns={"loss": "train_loss"})
    history_df["epoch"] = history_df["epoch"] + 1

    if "val_loss" in history_df.columns:
        best_row_idx = history_df["val_loss"].idxmin()
        best_val_loss = float(history_df.loc[best_row_idx, "val_loss"])
        best_epoch = int(history_df.loc[best_row_idx, "epoch"])
    else:
        best_val_loss = np.nan
        best_epoch = int(history_df["epoch"].max())

    candidate_rows.append({
        "candidate_id": candidate_id,
        "hidden_dims": str(hidden_dims),
        "dropout": dropout,
        "learning_rate": lr,
        "weight_decay": wd,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
    })

    candidate_artifacts.append({
        "candidate_id": candidate_id,
        "hidden_dims": hidden_dims,
        "dropout": dropout,
        "learning_rate": lr,
        "weight_decay": wd,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "history_df": history_df.copy(),
    })

tuning_results_df = pd.DataFrame(candidate_rows).sort_values(
    by=["best_val_loss", "candidate_id"],
    ascending=[True, True]
).reset_index(drop=True)

best_candidate = candidate_artifacts[
    int(tuning_results_df.iloc[0]["candidate_id"]) - 1
]

best_candidate_df = pd.DataFrame([{
    "candidate_id": best_candidate["candidate_id"],
    "hidden_dims": str(best_candidate["hidden_dims"]),
    "dropout": best_candidate["dropout"],
    "learning_rate": best_candidate["learning_rate"],
    "weight_decay": best_candidate["weight_decay"],
    "best_val_loss": best_candidate["best_val_loss"],
    "best_epoch": best_candidate["best_epoch"],
}])

# ------------------------------
# 7) Refit best model on full train
# ------------------------------
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

net_final = tt.practical.MLPVanilla(
    in_features=in_features,
    num_nodes=best_candidate["hidden_dims"],
    out_features=1,
    batch_norm=DEEPSURV_TUNING_CONFIG["batch_norm"],
    dropout=best_candidate["dropout"],
    output_bias=DEEPSURV_TUNING_CONFIG["output_bias"],
)

model_final = CoxPH(net_final, tt.optim.AdamW)
model_final.optimizer.set_lr(best_candidate["learning_rate"])

_ = model_final.fit(
    X_train_deepsurv,
    y_train_full,
    batch_size=DEEPSURV_TUNING_CONFIG["batch_size"],
    epochs=int(best_candidate["best_epoch"]),
    verbose=False,
)

model_final.compute_baseline_hazards()

best_history_df = best_candidate["history_df"].copy()

# ------------------------------
# 8) Predict survival on test
# ------------------------------
surv_df = model_final.predict_surv_df(X_test_deepsurv)
surv_df.columns = test_df_deepsurv["enrollment_id"].tolist()
surv_df.columns.name = "enrollment_id"
surv_df.index = surv_df.index.astype(float)

def get_surv_at_horizon(surv_frame: pd.DataFrame, h: int) -> pd.Series:
    idx = np.asarray(surv_frame.index, dtype=float)
    pos = np.searchsorted(idx, float(h), side="right") - 1
    if pos < 0:
        return pd.Series(np.ones(surv_frame.shape[1]), index=surv_frame.columns)
    return surv_frame.iloc[pos]

truth_test = pd.DataFrame({
    "enrollment_id": test_df_deepsurv["enrollment_id"].values,
    "event": event_test_deepsurv.astype(int),
    "duration": duration_test_deepsurv.astype(int),
})

truth_train = pd.DataFrame({
    "enrollment_id": train_df_deepsurv["enrollment_id"].values,
    "event": event_train_deepsurv.astype(int),
    "duration": duration_train_deepsurv.astype(int),
})

durations_test = truth_test["duration"].to_numpy(dtype=int)
events_test = truth_test["event"].to_numpy(dtype=int)

# ------------------------------
# 9) Primary survival metrics
# ------------------------------
eval_surv = EvalSurv(
    surv=surv_df,
    durations=durations_test,
    events=events_test,
    censor_surv="km",
)

primary_metrics_rows = []

try:
    c_index = float(eval_surv.concordance_td())
    c_index_note = "pycox_concordance_td"
except Exception as e:
    c_index = np.nan
    c_index_note = f"failed: {str(e)}"

primary_metrics_rows.append({
    "metric_name": "c_index",
    "metric_category": "co_primary",
    "metric_value": c_index,
    "notes": c_index_note,
})

try:
    max_requested_horizon = int(max(HORIZONS_WEEKS))
    ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
    ibs_value = float(eval_surv.integrated_brier_score(ibs_grid))
    ibs_note = "pycox_integrated_brier_score"
except Exception as e:
    try:
        max_requested_horizon = int(max(HORIZONS_WEEKS))
        ibs_grid = np.arange(1, max_requested_horizon + 1, dtype=int)
        dense_brier = eval_surv.brier_score(ibs_grid)
        ibs_value = float(np.trapz(dense_brier.values.astype(float), ibs_grid) / (ibs_grid[-1] - ibs_grid[0]))
        ibs_note = f"fallback_np_trapz_after_pycox_failure: {str(e)}"
    except Exception as e2:
        ibs_value = np.nan
        ibs_note = f"failed: {str(e)} | fallback_failed: {str(e2)}"

primary_metrics_rows.insert(0, {
    "metric_name": "ibs",
    "metric_category": "primary",
    "metric_value": ibs_value,
    "notes": ibs_note,
})

primary_metrics_summary = pd.DataFrame(primary_metrics_rows)

# ------------------------------
# 10) Brier by horizon
# ------------------------------
try:
    brier_h = eval_surv.brier_score(np.array(HORIZONS_WEEKS, dtype=int))
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": list(brier_h.index.astype(int)),
        "metric_name": ["brier_at_horizon"] * len(brier_h.index),
        "metric_category": ["primary"] * len(brier_h.index),
        "metric_value": list(brier_h.values.astype(float)),
        "notes": ["pycox_brier_score"] * len(brier_h.index),
    })
except Exception as e:
    brier_by_horizon_df = pd.DataFrame({
        "horizon_week": HORIZONS_WEEKS,
        "metric_name": ["brier_at_horizon"] * len(HORIZONS_WEEKS),
        "metric_category": ["primary"] * len(HORIZONS_WEEKS),
        "metric_value": [np.nan] * len(HORIZONS_WEEKS),
        "notes": [f"failed: {str(e)}"] * len(HORIZONS_WEEKS),
    })

# ------------------------------
# 11) Support, calibration, risk AUC, predicted vs observed
# ------------------------------
support_rows = []
risk_auc_rows = []
calibration_rows = []
pred_vs_obs_rows = []
test_pred_rows = []

for h in HORIZONS_WEEKS:
    pred_surv_h = get_surv_at_horizon(surv_df, h)
    pred_risk_h = 1.0 - pred_surv_h

    eval_df = truth_test.copy()
    eval_df["pred_survival_h"] = eval_df["enrollment_id"].map(pred_surv_h.to_dict())
    eval_df["pred_risk_h"] = eval_df["enrollment_id"].map(pred_risk_h.to_dict())

    eval_df["is_evaluable_at_h"] = (
        ((eval_df["event"] == 1) & (eval_df["duration"] <= h)) |
        (eval_df["duration"] >= h)
    ).astype(int)

    eval_df = eval_df[eval_df["is_evaluable_at_h"] == 1].copy()
    eval_df["observed_event_by_h"] = ((eval_df["event"] == 1) & (eval_df["duration"] <= h)).astype(int)
    eval_df["observed_survival_by_h"] = 1 - eval_df["observed_event_by_h"]

    support_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "n_events_by_horizon": int(eval_df["observed_event_by_h"].sum()),
        "event_rate_by_horizon": float(eval_df["observed_event_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df["observed_event_by_h"].nunique() >= 2:
        risk_auc = roc_auc_score(eval_df["observed_event_by_h"], eval_df["pred_risk_h"])
    else:
        risk_auc = np.nan

    risk_auc_rows.append({
        "horizon_week": h,
        "metric_name": "risk_auc_at_horizon",
        "metric_category": "secondary",
        "metric_value": float(risk_auc) if pd.notna(risk_auc) else np.nan,
        "notes": "roc_auc_on_evaluable_subset",
    })

    pred_vs_obs_rows.append({
        "horizon_week": h,
        "n_evaluable_enrollments": int(eval_df.shape[0]),
        "mean_predicted_survival": float(eval_df["pred_survival_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "mean_observed_survival": float(eval_df["observed_survival_by_h"].mean()) if eval_df.shape[0] > 0 else np.nan,
        "abs_gap": float(abs(eval_df["pred_survival_h"].mean() - eval_df["observed_survival_by_h"].mean())) if eval_df.shape[0] > 0 else np.nan,
    })

    if eval_df.shape[0] > 0:
        ranked = eval_df["pred_risk_h"].rank(method="first")
        n_bins_eff = int(min(CALIBRATION_BINS, max(1, eval_df.shape[0])))

        eval_df["calibration_bin"] = pd.qcut(
            ranked,
            q=n_bins_eff,
            labels=False,
            duplicates="drop"
        )

        calib_tab = (
            eval_df.groupby("calibration_bin")
            .agg(
                n=("enrollment_id", "count"),
                mean_predicted_risk=("pred_risk_h", "mean"),
                observed_event_rate=("observed_event_by_h", "mean"),
            )
            .reset_index()
        )
        calib_tab["horizon_week"] = h
        calib_tab["abs_calibration_gap"] = (
            calib_tab["mean_predicted_risk"] - calib_tab["observed_event_rate"]
        ).abs()

        for _, row in calib_tab.iterrows():
            calibration_rows.append({
                "horizon_week": int(row["horizon_week"]),
                "calibration_bin": int(row["calibration_bin"]),
                "n": int(row["n"]),
                "mean_predicted_risk": float(row["mean_predicted_risk"]),
                "observed_event_rate": float(row["observed_event_rate"]),
                "abs_calibration_gap": float(row["abs_calibration_gap"]),
            })

    tmp_preds = truth_test[["enrollment_id", "event", "duration"]].copy()
    tmp_preds["horizon_week"] = h
    tmp_preds["pred_survival_h"] = tmp_preds["enrollment_id"].map(pred_surv_h.to_dict())
    tmp_preds["pred_risk_h"] = tmp_preds["enrollment_id"].map(pred_risk_h.to_dict())
    test_pred_rows.append(tmp_preds)

support_by_horizon_df = pd.DataFrame(support_rows)
risk_auc_by_horizon_df = pd.DataFrame(risk_auc_rows)
predicted_vs_observed_survival_df = pd.DataFrame(pred_vs_obs_rows)
calibration_bins_df = pd.DataFrame(calibration_rows)
test_predictions_df = pd.concat(test_pred_rows, ignore_index=True)

calibration_summary_rows = []
for h in HORIZONS_WEEKS:
    h_df = calibration_bins_df[calibration_bins_df["horizon_week"] == h].copy()
    if h_df.shape[0] > 0:
        ece_like = np.average(h_df["abs_calibration_gap"], weights=h_df["n"])
    else:
        ece_like = np.nan
    calibration_summary_rows.append({
        "horizon_week": h,
        "metric_name": "calibration_at_horizon",
        "metric_category": "primary",
        "metric_value": float(ece_like) if pd.notna(ece_like) else np.nan,
        "notes": "Weighted absolute calibration gap across bins",
    })

calibration_summary_df = pd.DataFrame(calibration_summary_rows)

# ------------------------------
# 12) Optional time-dependent AUC
# ------------------------------
td_auc_rows = []
if SKSURV_AVAILABLE:
    try:
        y_train_surv = Surv.from_arrays(
            event=truth_train["event"].astype(bool).to_numpy(),
            time=truth_train["duration"].astype(float).to_numpy(),
        )
        y_test_surv = Surv.from_arrays(
            event=truth_test["event"].astype(bool).to_numpy(),
            time=truth_test["duration"].astype(float).to_numpy(),
        )

        for h in HORIZONS_WEEKS:
            pred_risk_h = 1.0 - get_surv_at_horizon(surv_df, h)
            auc_vals, mean_auc = cumulative_dynamic_auc(
                y_train_surv,
                y_test_surv,
                pred_risk_h.to_numpy(),
                np.array([h], dtype=float)
            )
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": float(auc_vals[0]),
                "notes": "sksurv cumulative_dynamic_auc",
            })
    except Exception as e:
        for h in HORIZONS_WEEKS:
            td_auc_rows.append({
                "horizon_week": h,
                "metric_name": "time_dependent_auc",
                "metric_category": "secondary",
                "metric_value": np.nan,
                "notes": f"sksurv_available_but_failed: {str(e)}",
            })
else:
    for h in HORIZONS_WEEKS:
        td_auc_rows.append({
            "horizon_week": h,
            "metric_name": "time_dependent_auc",
            "metric_category": "secondary",
            "metric_value": np.nan,
            "notes": "sksurv_unavailable",
        })

time_dependent_auc_df = pd.DataFrame(td_auc_rows)
secondary_metrics_df = pd.concat([risk_auc_by_horizon_df, time_dependent_auc_df], ignore_index=True)

# ------------------------------
# 13) Row-level diagnostics (auxiliary proxy only)
# ------------------------------
test_event_times = truth_test["duration"].astype(int).tolist()
row_pred_surv = []
for eid, h in zip(truth_test["enrollment_id"], test_event_times):
    row_pred_surv.append(float(get_surv_at_horizon(surv_df, h).get(eid, np.nan)))
row_pred_surv = np.asarray(row_pred_surv)
row_pred_risk = 1.0 - row_pred_surv

valid_mask = np.isfinite(row_pred_risk)
y_true_row = truth_test.loc[valid_mask, "event"].to_numpy().astype(int)
y_prob_row = np.clip(row_pred_risk[valid_mask], 1e-8, 1 - 1e-8)

if len(np.unique(y_true_row)) >= 2:
    row_roc_auc = roc_auc_score(y_true_row, y_prob_row)
    row_pr_auc = average_precision_score(y_true_row, y_prob_row)
else:
    row_roc_auc = np.nan
    row_pr_auc = np.nan

row_log_loss = log_loss(y_true_row, y_prob_row, labels=[0, 1]) if len(y_true_row) > 0 else np.nan
row_brier = np.mean((y_prob_row - y_true_row) ** 2) if len(y_true_row) > 0 else np.nan

row_diagnostics_df = pd.DataFrame([{
    "model_name": "deepsurv_tuned",
    "row_level_roc_auc": float(row_roc_auc) if pd.notna(row_roc_auc) else np.nan,
    "row_level_pr_auc": float(row_pr_auc) if pd.notna(row_pr_auc) else np.nan,
    "row_level_log_loss": float(row_log_loss) if pd.notna(row_log_loss) else np.nan,
    "row_level_brier": float(row_brier) if pd.notna(row_brier) else np.nan,
    "diagnostic_note": "auxiliary proxy only; not a primary cross-family benchmark metric",
}])

# ------------------------------
# 14) Save artifacts
# ------------------------------
model_dir = OUTPUT_DIR / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "deepsurv_tuned.pt"
preprocessor_path = model_dir / "deepsurv_preprocessor.joblib"
tuning_results_path = TABLES_DIR / "table_deepsurv_tuning_results.csv"
training_history_path = TABLES_DIR / "table_deepsurv_tuned_training_history.csv"
predictions_path = TABLES_DIR / "table_deepsurv_tuned_test_predictions.csv"
primary_metrics_path = TABLES_DIR / "table_deepsurv_tuned_primary_metrics.csv"
brier_path = TABLES_DIR / "table_deepsurv_tuned_brier_by_horizon.csv"
secondary_metrics_path = TABLES_DIR / "table_deepsurv_tuned_secondary_metrics.csv"
row_diag_path = TABLES_DIR / "table_deepsurv_tuned_row_diagnostics.csv"
support_path = TABLES_DIR / "table_deepsurv_tuned_support_by_horizon.csv"
cal_summary_path = TABLES_DIR / "table_deepsurv_tuned_calibration_summary.csv"
cal_bins_path = TABLES_DIR / "table_deepsurv_tuned_calibration_bins_by_horizon.csv"
pred_vs_obs_path = TABLES_DIR / "table_deepsurv_tuned_predicted_vs_observed_survival.csv"
config_path = METADATA_DIR / "deepsurv_tuned_model_config.json"

torch.save(model_final.net.state_dict(), model_path)
joblib.dump(deepsurv_preprocessor, preprocessor_path)

tuning_results_df.to_csv(tuning_results_path, index=False)
best_history_df.to_csv(training_history_path, index=False)
test_predictions_df.to_csv(predictions_path, index=False)
primary_metrics_summary.to_csv(primary_metrics_path, index=False)
brier_by_horizon_df.to_csv(brier_path, index=False)
secondary_metrics_df.to_csv(secondary_metrics_path, index=False)
row_diagnostics_df.to_csv(row_diag_path, index=False)
support_by_horizon_df.to_csv(support_path, index=False)
calibration_summary_df.to_csv(cal_summary_path, index=False)
calibration_bins_df.to_csv(cal_bins_path, index=False)
predicted_vs_observed_survival_df.to_csv(pred_vs_obs_path, index=False)

tuned_config_to_save = copy.deepcopy(DEEPSURV_TUNING_CONFIG)
tuned_config_to_save["best_candidate"] = {
    "candidate_id": int(best_candidate["candidate_id"]),
    "hidden_dims": best_candidate["hidden_dims"],
    "dropout": float(best_candidate["dropout"]),
    "learning_rate": float(best_candidate["learning_rate"]),
    "weight_decay": float(best_candidate["weight_decay"]),
    "best_val_loss": float(best_candidate["best_val_loss"]),
    "best_epoch": int(best_candidate["best_epoch"]),
}
save_json(tuned_config_to_save, config_path)

# ------------------------------
# 15) Output for feedback
# ------------------------------
print("\nTuning results:")
display(tuning_results_df)

print("\nBest candidate:")
display(best_candidate_df)

print("\nPrimary metrics summary:")
display(primary_metrics_summary)

print("\nBrier by horizon:")
display(brier_by_horizon_df)

print("\nCalibration summary by horizon:")
display(calibration_summary_df)

print("\nSecondary metrics:")
display(secondary_metrics_df)

print("\nRow-level diagnostics:")
display(row_diagnostics_df)

print("\nSupport by horizon:")
display(support_by_horizon_df)

print("\nPredicted vs observed survival:")
display(predicted_vs_observed_survival_df)

print("\nSaved artifacts:")
print("-", model_path.resolve())
print("-", preprocessor_path.resolve())
print("-", tuning_results_path.resolve())
print("-", training_history_path.resolve())
print("-", predictions_path.resolve())
print("-", primary_metrics_path.resolve())
print("-", brier_path.resolve())
print("-", secondary_metrics_path.resolve())
print("-", row_diag_path.resolve())
print("-", support_path.resolve())
print("-", cal_summary_path.resolve())
print("-", cal_bins_path.resolve())
print("-", pred_vs_obs_path.resolve())
print("-", config_path.resolve())

## D6 — Consolidated Benchmark Comparison and Leaderboard


### Funcao: read_metric_value

Definicao isolada de read_metric_value para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D6 — Consolidated Benchmark Comparison and Leaderboard
# --------------------------------------------------------------
# Purpose:
#   Consolidate the benchmark outputs from the selected official
#   model variants and build the main leaderboard tables.
#
# Inputs expected from previous cells:
#   - OUTPUT_DIR
#   - tables generated by the official selected variants
#
# Official variants consolidated here:
#   - linear_tuned
#   - neural_tuned
#   - cox_tuned
#   - deepsurv_tuned
#
# Outputs:
#   - table_benchmark_leaderboard_main.csv
#   - table_benchmark_leaderboard_by_horizon.csv
#   - table_benchmark_selected_sources.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6 — Consolidated Benchmark Comparison and Leaderboard")
print("=" * 70)

print("Methodological note: this step consolidates benchmark outputs only.")
print("No model is trained here.")
print("This version uses the official selected model variants explicitly.")

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------
# 1) Basic paths
# ------------------------------
required_names = ["OUTPUT_DIR", "TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

OUT_TABLES = TABLES_DIR
OUT_TABLES.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Official source registry
# ------------------------------
MODEL_SOURCES = {
    "linear_tuned": {
        "display_name": "Linear Discrete-Time (Tuned)",
        "family": "discrete_time_linear",
        "primary": OUT_TABLES / "table_linear_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_linear_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_linear_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_linear_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_linear_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_linear_tuned_predicted_vs_observed_survival.csv",
    },
    "neural_tuned": {
        "display_name": "Neural Discrete-Time (Tuned)",
        "family": "discrete_time_neural",
        "primary": OUT_TABLES / "table_neural_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_neural_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_neural_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_neural_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_neural_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_neural_tuned_predicted_vs_observed_survival.csv",
    },
    "cox_tuned": {
        "display_name": "Cox (Tuned)",
        "family": "continuous_time_cox",
        "primary": OUT_TABLES / "table_cox_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_cox_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_cox_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_cox_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_cox_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_cox_tuned_predicted_vs_observed_survival.csv",
    },
    "deepsurv_tuned": {
        "display_name": "DeepSurv (Tuned)",
        "family": "continuous_time_neural",
        "primary": OUT_TABLES / "table_deepsurv_tuned_primary_metrics.csv",
        "brier": OUT_TABLES / "table_deepsurv_tuned_brier_by_horizon.csv",
        "calibration": OUT_TABLES / "table_deepsurv_tuned_calibration_summary.csv",
        "secondary": OUT_TABLES / "table_deepsurv_tuned_secondary_metrics.csv",
        "support": OUT_TABLES / "table_deepsurv_tuned_support_by_horizon.csv",
        "pred_vs_obs": OUT_TABLES / "table_deepsurv_tuned_predicted_vs_observed_survival.csv",
    },
}

# ------------------------------
# 3) Validate files
# ------------------------------
missing_files = []
selected_source_rows = []

for model_key, cfg in MODEL_SOURCES.items():
    for artifact_name, path in cfg.items():
        if artifact_name in ["display_name", "family"]:
            continue
        exists = path.exists()
        selected_source_rows.append({
            "model_key": model_key,
            "display_name": cfg["display_name"],
            "family": cfg["family"],
            "artifact_name": artifact_name,
            "path": str(path),
            "exists": exists,
        })
        if not exists:
            missing_files.append(str(path))

selected_sources_df = pd.DataFrame(selected_source_rows)

if missing_files:
    raise FileNotFoundError(
        "Some benchmark result files are missing for the selected official variants:\n- "
        + "\n- ".join(missing_files)
    )

# ------------------------------
# 4) Read helper functions
# ------------------------------
def read_metric_value(primary_path: Path, metric_name: str):
    df = pd.read_csv(primary_path)
    row = df.loc[df["metric_name"] == metric_name]
    if row.empty:
        return np.nan
    return float(row["metric_value"].iloc[0])

# ------------------------------
# 5) Main leaderboard
# ------------------------------
leader_rows = []

for model_key, cfg in MODEL_SOURCES.items():
    ibs = read_metric_value(cfg["primary"], "ibs")
    c_index = read_metric_value(cfg["primary"], "c_index")

    calib_df = pd.read_csv(cfg["calibration"])
    mean_calibration_gap = float(calib_df["metric_value"].mean()) if not calib_df.empty else np.nan

    leader_rows.append({
        "model_key": model_key,
        "display_name": cfg["display_name"],
        "family": cfg["family"],
        "ibs": ibs,
        "c_index": c_index,
        "mean_calibration_gap": mean_calibration_gap,
    })

leaderboard_main_df = pd.DataFrame(leader_rows)

leaderboard_main_df["rank_ibs"] = leaderboard_main_df["ibs"].rank(method="min", ascending=True)
leaderboard_main_df["rank_c_index"] = leaderboard_main_df["c_index"].rank(method="min", ascending=False)
leaderboard_main_df["rank_mean_calibration_gap"] = leaderboard_main_df["mean_calibration_gap"].rank(method="min", ascending=True)

leaderboard_main_df = leaderboard_main_df.sort_values(
    ["rank_ibs", "rank_mean_calibration_gap", "rank_c_index", "display_name"]
).reset_index(drop=True)

# ------------------------------
# 6) Horizon leaderboard
# ------------------------------
horizon_rows = []

for model_key, cfg in MODEL_SOURCES.items():
    brier_df = pd.read_csv(cfg["brier"]).copy()
    calib_df = pd.read_csv(cfg["calibration"]).copy()
    support_df = pd.read_csv(cfg["support"]).copy()
    pred_obs_df = pd.read_csv(cfg["pred_vs_obs"]).copy()

    brier_df = brier_df.rename(columns={"metric_value": "brier_value"})
    calib_df = calib_df.rename(columns={"metric_value": "calibration_gap"})
    pred_obs_df = pred_obs_df.rename(columns={"abs_gap": "predicted_vs_observed_abs_gap"})

    merged = (
        brier_df[["horizon_week", "brier_value"]]
        .merge(calib_df[["horizon_week", "calibration_gap"]], on="horizon_week", how="outer")
        .merge(
            support_df[["horizon_week", "n_evaluable_enrollments", "n_events_by_horizon", "event_rate_by_horizon"]],
            on="horizon_week",
            how="left"
        )
        .merge(
            pred_obs_df[["horizon_week", "predicted_vs_observed_abs_gap"]],
            on="horizon_week",
            how="left"
        )
    )

    merged["model_key"] = model_key
    merged["display_name"] = cfg["display_name"]
    merged["family"] = cfg["family"]

    horizon_rows.append(merged)

leaderboard_horizon_df = pd.concat(horizon_rows, ignore_index=True)

leaderboard_horizon_df = leaderboard_horizon_df[
    [
        "model_key",
        "display_name",
        "family",
        "horizon_week",
        "brier_value",
        "calibration_gap",
        "predicted_vs_observed_abs_gap",
        "n_evaluable_enrollments",
        "n_events_by_horizon",
        "event_rate_by_horizon",
    ]
].sort_values(["horizon_week", "display_name"]).reset_index(drop=True)

# ------------------------------
# 7) Save outputs
# ------------------------------
selected_sources_path = OUT_TABLES / "table_benchmark_selected_sources.csv"
leaderboard_main_path = OUT_TABLES / "table_benchmark_leaderboard_main.csv"
leaderboard_horizon_path = OUT_TABLES / "table_benchmark_leaderboard_by_horizon.csv"

selected_sources_df.to_csv(selected_sources_path, index=False)
leaderboard_main_df.to_csv(leaderboard_main_path, index=False)
leaderboard_horizon_df.to_csv(leaderboard_horizon_path, index=False)

# ------------------------------
# 8) Output
# ------------------------------
print("\nSelected official benchmark sources:")
display(selected_sources_df)

print("\nMain leaderboard:")
display(leaderboard_main_df)

print("\nLeaderboard by horizon:")
display(leaderboard_horizon_df)

print("\nSaved:")
print("-", selected_sources_path.resolve())
print("-", leaderboard_main_path.resolve())
print("-", leaderboard_horizon_path.resolve())

### D6.1 — Define Expanded Survival Calibration Diagnostics


### Funcao: _safe_read_csv

Definicao isolada de _safe_read_csv para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# D6.1 — Define Expanded Survival Calibration Diagnostics
# --------------------------------------------------------------
# Purpose:
#   Formalize the expanded calibration protocol used by the benchmark,
#   including:
#     - eligible models
#     - official benchmark horizons
#     - primary calibration metric
#     - required strengthening diagnostics
#     - support / comparability rules
#     - required downstream outputs for P26.5 and P26.6
#
# Design principle:
#   This is a protocol-definition step, not a metric-recomputation step.
#   It creates the canonical specification that later cells must follow.
#
# Notes:
#   - The benchmark already uses shared horizons 10/20/30.
#   - The benchmark already uses a horizon-specific weighted absolute
#     calibration gap across bins as the main calibration metric.
#   - The strengthened layer now explicitly requires approximate
#     calibration intercept and slope by horizon.
#   - An ICI-style metric is kept as optional / non-blocking unless
#     row-level predictions are available in a stable canonical form.
# ==============================================================

print("=" * 70)
print("D6.1 — Define Expanded Survival Calibration Diagnostics")
print("=" * 70)

# ------------------------------
# 0) Dependency checks
# ------------------------------
required_names = ["TABLES_DIR", "METADATA_DIR", "save_json"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

from pathlib import Path
import json
import pandas as pd
from datetime import datetime

# Optional objects
duckdb_available = "con" in globals()

# ------------------------------
# 1) Benchmark horizons
# ------------------------------
# Prefer an existing notebook/global convention if present.
if "BENCHMARK_HORIZONS" in globals():
    benchmark_horizons = [int(x) for x in BENCHMARK_HORIZONS]
elif "HORIZON_WEEKS" in globals():
    benchmark_horizons = [int(x) for x in HORIZON_WEEKS]
else:
    benchmark_horizons = [10, 20, 30]

benchmark_horizons = sorted(list(dict.fromkeys(benchmark_horizons)))

# ------------------------------
# 2) Discover currently available calibration artifacts
# ------------------------------
table_dir = Path(TABLES_DIR)
metadata_dir = Path(METADATA_DIR)

artifact_candidates = {
    "benchmark_calibration_by_horizon_wide": table_dir / "table_benchmark_calibration_by_horizon_wide.csv",
    "all_tuned_calibration_audit": table_dir / "table_p26_5_all_tuned_calibration_audit.csv",
    "paper_summary_all_tuned_models": table_dir / "table_p26_5_paper_summary_all_tuned_models.csv",
    "reliability_bins_all_tuned_models": table_dir / "table_p26_5_reliability_bins_all_tuned_models.csv",
    "reliability_bins_h20_all_tuned_models": table_dir / "table_p26_5_reliability_bins_h20_all_tuned_models.csv",
    "p11_1_slope_intercept_main": table_dir / "table_p11_1_calibration_slope_intercept_by_horizon.csv",
    "p11_1_slope_intercept_summary": table_dir / "table_p11_1_calibration_slope_intercept_summary.csv",
    "p11_2_unified_strengthening": table_dir / "table_p11_2_unified_calibration_strengthening.csv",
}

artifact_inventory_rows = []
for artifact_name, artifact_path in artifact_candidates.items():
    artifact_inventory_rows.append({
        "artifact_name": artifact_name,
        "exists": artifact_path.exists(),
        "path": artifact_path.as_posix(),
    })

artifact_inventory_df = pd.DataFrame(artifact_inventory_rows)

# ------------------------------
# 3) Infer eligible model families, if possible
# ------------------------------
def _safe_read_csv(path_obj):
    try:
        return pd.read_csv(path_obj)
    except Exception:
        return None

eligible_models_inferred = []

# First preference: existing all-tuned audit
p26_5_audit_path = artifact_candidates["all_tuned_calibration_audit"]
if p26_5_audit_path.exists():
    df_tmp = _safe_read_csv(p26_5_audit_path)
    if df_tmp is not None:
        for candidate_col in ["model_label", "model_name", "model_id", "family_label", "family_name"]:
            if candidate_col in df_tmp.columns:
                vals = (
                    df_tmp[candidate_col]
                    .dropna()
                    .astype(str)
                    .str.strip()
                    .tolist()
                )
                eligible_models_inferred = sorted(list(dict.fromkeys([v for v in vals if v])))
                if eligible_models_inferred:
                    break

# Second preference: unified strengthening table
if not eligible_models_inferred:
    p11_2_path = artifact_candidates["p11_2_unified_strengthening"]
    if p11_2_path.exists():
        df_tmp = _safe_read_csv(p11_2_path)
        if df_tmp is not None:
            for candidate_col in ["model_label", "model_name", "model_id", "family_label", "family_name"]:
                if candidate_col in df_tmp.columns:
                    vals = (
                        df_tmp[candidate_col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    eligible_models_inferred = sorted(list(dict.fromkeys([v for v in vals if v])))
                    if eligible_models_inferred:
                        break

# Conservative fallback if nothing canonical is found yet
if not eligible_models_inferred:
    eligible_models_inferred = [
        "cox_tuned",
        "deepsurv_tuned",
        "linear_discrete_time_tuned",
        "neural_discrete_time_tuned",
    ]

# ------------------------------
# 4) Define expanded calibration protocol
# ------------------------------
protocol_rows = [
    {
        "protocol_component": "target_scope",
        "value": "all_tuned_benchmark_families",
        "status": "required",
        "notes": (
            "Expanded calibration diagnostics apply to tuned benchmark families "
            "that produce enrollment-level risk or survival outputs at shared horizons."
        ),
    },
    {
        "protocol_component": "eligible_models_rule",
        "value": "tuned_models_with_horizon_level_predictions_and_reliability_bins",
        "status": "required",
        "notes": (
            "A model is eligible if it has stable test-set horizon predictions and can be "
            "represented in the calibration audit at the shared benchmark horizons."
        ),
    },
    {
        "protocol_component": "shared_horizons",
        "value": ",".join(map(str, benchmark_horizons)),
        "status": "required",
        "notes": "Official benchmark horizons for calibration comparison.",
    },
    {
        "protocol_component": "primary_metric_name",
        "value": "weighted_absolute_calibration_gap_by_horizon",
        "status": "required",
        "notes": (
            "Primary calibration metric. Computed as the sample-size-weighted mean absolute "
            "gap between mean predicted risk and empirical event rate across risk bins."
        ),
    },
    {
        "protocol_component": "primary_metric_role",
        "value": "main_calibration_ranking_signal",
        "status": "required",
        "notes": (
            "Lower values indicate better alignment between predicted event risk and "
            "observed event frequency at a given horizon."
        ),
    },
    {
        "protocol_component": "required_strengthening_metric_1",
        "value": "approx_calibration_intercept_by_horizon",
        "status": "required",
        "notes": (
            "Approximate intercept derived from weighted fit on reliability bins. "
            "Used to assess systematic under/over-prediction."
        ),
    },
    {
        "protocol_component": "required_strengthening_metric_2",
        "value": "approx_calibration_slope_by_horizon",
        "status": "required",
        "notes": (
            "Approximate slope derived from weighted fit on reliability bins. "
            "Used to assess spread/compression of predicted risk."
        ),
    },
    {
        "protocol_component": "secondary_metric_1",
        "value": "brier_at_horizon",
        "status": "required",
        "notes": (
            "Retained as complementary performance metric at shared horizons; "
            "not a substitute for calibration-specific diagnostics."
        ),
    },
    {
        "protocol_component": "secondary_metric_2",
        "value": "support_fraction_or_n_at_horizon",
        "status": "required",
        "notes": (
            "Used to determine whether a horizon remains interpretable and comparable."
        ),
    },
    {
        "protocol_component": "secondary_metric_3",
        "value": "event_rate_at_horizon",
        "status": "required",
        "notes": (
            "Provides outcome prevalence context for calibration interpretation."
        ),
    },
    {
        "protocol_component": "secondary_metric_4",
        "value": "reliability_bins_table",
        "status": "required",
        "notes": (
            "Canonical artifact backing calibration plots and bin-level audit."
        ),
    },
    {
        "protocol_component": "optional_metric",
        "value": "ici_style_summary_if_row_level_predictions_are_stably_available",
        "status": "optional_non_blocking",
        "notes": (
            "ICI-style summary is desirable, but not required for the canonical benchmark "
            "if row-level predictions are not yet available in a stable unified structure."
        ),
    },
    {
        "protocol_component": "support_rule",
        "value": "report_all_horizons_but_flag_low_support",
        "status": "required",
        "notes": (
            "Horizons should be reported, but low-support horizons must be explicitly flagged "
            "and should not dominate comparative interpretation."
        ),
    },
    {
        "protocol_component": "comparability_rule",
        "value": "compare_models_only_at_shared_horizons_with_available_predictions",
        "status": "required",
        "notes": (
            "Cross-family comparison must remain restricted to shared benchmark horizons "
            "where all compared models have valid outputs."
        ),
    },
    {
        "protocol_component": "estimation_rule_for_slope_intercept",
        "value": "weighted_logit_scale_fit_over_reliability_bins",
        "status": "required",
        "notes": (
            "Approximate calibration slope/intercept are estimated through a weighted fit "
            "over reliability-bin summaries, not through a separate retraining workflow."
        ),
    },
    {
        "protocol_component": "interpretation_rule",
        "value": "use_multi_metric_calibration_reading",
        "status": "required",
        "notes": (
            "Final calibration interpretation must jointly consider primary gap, "
            "slope/intercept, support, and Brier; no single scalar should dominate the narrative."
        ),
    },
]

protocol_df = pd.DataFrame(protocol_rows)

# ------------------------------
# 5) Register eligible models
# ------------------------------
eligible_models_df = pd.DataFrame({
    "model_label": eligible_models_inferred,
    "eligibility_status": "eligible_if_horizon_outputs_present",
    "eligibility_basis": (
        "Tuned benchmark family with expected horizon-level calibration audit compatibility."
    ),
})

# ------------------------------
# 6) Define required downstream outputs
# ------------------------------
downstream_outputs_rows = [
    {
        "downstream_step": "P26.5",
        "required_output_name": "table_p26_5_all_tuned_calibration_audit.csv",
        "required": True,
        "purpose": (
            "Long-form audit table with one row per model-horizon-metric combination."
        ),
    },
    {
        "downstream_step": "P26.5",
        "required_output_name": "table_p26_5_paper_summary_all_tuned_models.csv",
        "required": True,
        "purpose": (
            "Paper-facing compact summary of calibration across tuned benchmark families."
        ),
    },
    {
        "downstream_step": "P26.5",
        "required_output_name": "table_p26_5_reliability_bins_all_tuned_models.csv",
        "required": True,
        "purpose": (
            "Canonical reliability-bin table supporting bin-level calibration audit."
        ),
    },
    {
        "downstream_step": "P26.5",
        "required_output_name": "figure_p26_5_reliability_selected_horizon.png",
        "required": False,
        "purpose": (
            "Optional visualization for selected horizon(s), especially h20 or paper-chosen horizon."
        ),
    },
    {
        "downstream_step": "P26.6",
        "required_output_name": "table_p26_6_calibration_interpretation_summary.csv",
        "required": True,
        "purpose": (
            "Narrative-ready interpretation layer summarizing comparative calibration strengths/weaknesses."
        ),
    },
]

downstream_outputs_df = pd.DataFrame(downstream_outputs_rows)

# ------------------------------
# 7) Define metric registry
# ------------------------------
metric_registry_rows = [
    {
        "metric_name": "weighted_absolute_calibration_gap_by_horizon",
        "metric_family": "calibration",
        "metric_role": "primary",
        "required": True,
        "lower_is_better": True,
        "estimation_level": "horizon",
        "canonical_source": "reliability_bins",
        "notes": (
            "Sample-size-weighted mean absolute gap between predicted risk and observed event rate across bins."
        ),
    },
    {
        "metric_name": "approx_calibration_intercept_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "required_secondary",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "weighted_fit_on_reliability_bins",
        "notes": (
            "Approximate intercept; closeness to 0 is desirable."
        ),
    },
    {
        "metric_name": "approx_calibration_slope_by_horizon",
        "metric_family": "calibration_strengthening",
        "metric_role": "required_secondary",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "weighted_fit_on_reliability_bins",
        "notes": (
            "Approximate slope; closeness to 1 is desirable."
        ),
    },
    {
        "metric_name": "brier_at_horizon",
        "metric_family": "overall_prediction_quality",
        "metric_role": "supporting",
        "required": True,
        "lower_is_better": True,
        "estimation_level": "horizon",
        "canonical_source": "benchmark_outputs",
        "notes": (
            "Complementary performance metric, not a direct substitute for calibration."
        ),
    },
    {
        "metric_name": "support_fraction_or_n_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "supporting",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "benchmark_outputs",
        "notes": (
            "Used to qualify interpretability and comparability of horizon-level summaries."
        ),
    },
    {
        "metric_name": "event_rate_at_horizon",
        "metric_family": "audit_context",
        "metric_role": "supporting",
        "required": True,
        "lower_is_better": False,
        "estimation_level": "horizon",
        "canonical_source": "benchmark_outputs_or_bins",
        "notes": (
            "Outcome prevalence context for horizon-level interpretation."
        ),
    },
    {
        "metric_name": "ici_style_summary",
        "metric_family": "calibration_strengthening",
        "metric_role": "optional_exploratory",
        "required": False,
        "lower_is_better": True,
        "estimation_level": "horizon_or_global",
        "canonical_source": "row_level_predictions_if_available",
        "notes": (
            "Optional. Implement only if row-level predictions are available in stable canonical form."
        ),
    },
]

metric_registry_df = pd.DataFrame(metric_registry_rows)

# ------------------------------
# 8) Save outputs
# ------------------------------
path_protocol = table_dir / "table_p26_4_expanded_calibration_protocol.csv"
path_eligible = table_dir / "table_p26_4_calibration_eligible_models.csv"
path_metric_registry = table_dir / "table_p26_4_calibration_metric_registry.csv"
path_artifacts = table_dir / "table_p26_4_existing_calibration_artifacts.csv"
path_downstream = table_dir / "table_p26_4_required_downstream_outputs.csv"
path_metadata = metadata_dir / "metadata_p26_4_expanded_calibration_protocol.json"

protocol_df.to_csv(path_protocol, index=False)
eligible_models_df.to_csv(path_eligible, index=False)
metric_registry_df.to_csv(path_metric_registry, index=False)
artifact_inventory_df.to_csv(path_artifacts, index=False)
downstream_outputs_df.to_csv(path_downstream, index=False)

metadata = {
    "step": "D6.1",
    "title": "Define Expanded Survival Calibration Diagnostics",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "benchmark_horizons": benchmark_horizons,
    "eligible_models_inferred": eligible_models_inferred,
    "protocol_design_note": (
        "This step formalizes the expanded calibration protocol but does not recompute metrics."
    ),
    "primary_metric": "weighted_absolute_calibration_gap_by_horizon",
    "required_strengthening_metrics": [
        "approx_calibration_intercept_by_horizon",
        "approx_calibration_slope_by_horizon",
    ],
    "optional_metric": "ici_style_summary",
    "support_rule": "report_all_horizons_but_flag_low_support",
    "comparability_rule": "compare_models_only_at_shared_horizons_with_available_predictions",
    "required_downstream_outputs": downstream_outputs_df["required_output_name"].tolist(),
    "artifact_inventory": artifact_inventory_rows,
}

save_json(metadata, path_metadata)

# ------------------------------
# 9) Console summary
# ------------------------------
print("\nOfficial benchmark horizons:")
print(benchmark_horizons)

print("\nEligible models (inferred / fallback):")
display(eligible_models_df)

print("\nExpanded calibration protocol:")
display(protocol_df)

print("\nCalibration metric registry:")
display(metric_registry_df)

print("\nExisting calibration artifacts detected:")
display(artifact_inventory_df)

print("\nRequired downstream outputs:")
display(downstream_outputs_df)

print("\nSaved:")
print("-", path_protocol.resolve())
print("-", path_eligible.resolve())
print("-", path_metric_registry.resolve())
print("-", path_artifacts.resolve())
print("-", path_downstream.resolve())
print("-", path_metadata.resolve())

#### D6.1.1 — Materialize Benchmark Wide Comparison Tables

In [ ]:
# ==============================================================
# D6.1.1 — Materialize Benchmark Wide Comparison Tables
# --------------------------------------------------------------
# Purpose:
#   Build the wide-format benchmark comparison tables required
#   by downstream expanded calibration audit steps.
#
# Inputs expected from previous cells:
#   - TABLES_DIR
#   - table_benchmark_leaderboard_by_horizon.csv
#
# Outputs:
#   - table_benchmark_calibration_by_horizon_wide.csv
#   - table_benchmark_brier_by_horizon_wide.csv
#   - table_benchmark_support_reference.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6.1.1 — Materialize Benchmark Wide Comparison Tables")
print("=" * 70)

from pathlib import Path
import pandas as pd

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

leaderboard_horizon_path = TABLES_DIR / "table_benchmark_leaderboard_by_horizon.csv"
if not leaderboard_horizon_path.exists():
    raise FileNotFoundError(
        f"Missing required file: {leaderboard_horizon_path}. "
        "Run D6 first."
    )

# ------------------------------
# 2) Read benchmark by-horizon table
# ------------------------------
df = pd.read_csv(leaderboard_horizon_path)

required_cols = [
    "model_key",
    "display_name",
    "family",
    "horizon_week",
    "brier_value",
    "calibration_gap",
    "n_evaluable_enrollments",
    "n_events_by_horizon",
    "event_rate_by_horizon",
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(
        "Benchmark leaderboard by horizon is missing required columns: "
        + ", ".join(missing_cols)
    )

# ------------------------------
# 3) Calibration wide
# ------------------------------
calibration_wide_df = (
    df.pivot_table(
        index=["model_key", "display_name", "family"],
        columns="horizon_week",
        values="calibration_gap",
        aggfunc="first"
    )
    .reset_index()
)

calibration_wide_df.columns = [
    f"calibration_h{int(c)}" if isinstance(c, (int, float)) else c
    for c in calibration_wide_df.columns
]

# ------------------------------
# 4) Brier wide
# ------------------------------
brier_wide_df = (
    df.pivot_table(
        index=["model_key", "display_name", "family"],
        columns="horizon_week",
        values="brier_value",
        aggfunc="first"
    )
    .reset_index()
)

brier_wide_df.columns = [
    f"brier_h{int(c)}" if isinstance(c, (int, float)) else c
    for c in brier_wide_df.columns
]

# ------------------------------
# 5) Support reference
# ------------------------------
support_reference_df = (
    df[
        [
            "horizon_week",
            "n_evaluable_enrollments",
            "n_events_by_horizon",
            "event_rate_by_horizon",
        ]
    ]
    .drop_duplicates()
    .sort_values(["horizon_week", "n_evaluable_enrollments"], ascending=[True, False])
    .reset_index(drop=True)
)

# ------------------------------
# 6) Save outputs
# ------------------------------
calibration_wide_path = TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"
brier_wide_path = TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"
support_reference_path = TABLES_DIR / "table_benchmark_support_reference.csv"

calibration_wide_df.to_csv(calibration_wide_path, index=False)
brier_wide_df.to_csv(brier_wide_path, index=False)
support_reference_df.to_csv(support_reference_path, index=False)

# ------------------------------
# 7) Output
# ------------------------------
print("\nCalibration wide table:")
display(calibration_wide_df)

print("\nBrier wide table:")
display(brier_wide_df)

print("\nSupport reference:")
display(support_reference_df)

print("\nSaved:")
print("-", calibration_wide_path.resolve())
print("-", brier_wide_path.resolve())
print("-", support_reference_path.resolve())

### D6.2 — Unified Expanded Calibration Audit Across Tuned Models


In [ ]:
# ==============================================================
# D6.2 — Unified Expanded Calibration Audit Across Tuned Models
# --------------------------------------------------------------
# Purpose:
#   Build a unified expanded calibration audit across the tuned
#   benchmark models using the canonical benchmark-wide tables.
#
# Design:
#   - benchmark-wide calibration, brier, and support tables are required
#   - slope/intercept artifacts from P11.1 are optional if not yet present
#   - legacy reliability tables are optional
#
# Outputs:
#   - table_p26_5_all_tuned_calibration_audit.csv
#   - table_p26_5_paper_summary_all_tuned_models.csv
#   - table_p26_5_reliability_bins_all_tuned_models.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6.2 — Unified Expanded Calibration Audit Across Tuned Models")
print("=" * 70)

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 2) Paths
# ------------------------------
benchmark_calibration_path = TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"
benchmark_brier_path = TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"
support_reference_path = TABLES_DIR / "table_benchmark_support_reference.csv"

p11_1_by_horizon_path = TABLES_DIR / "table_p11_1_calibration_slope_intercept_by_horizon.csv"
legacy_reliability_all_path = TABLES_DIR / "table_calibration_reliability_bins_all_tuned_models.csv"
legacy_reliability_h20_path = TABLES_DIR / "table_calibration_reliability_bins_h20_all_tuned_models.csv"


### Funcao: read_required_csv

Definicao isolada de read_required_csv para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 3) Read helpers
# ------------------------------
def read_required_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file for {label}: {path}")
    return pd.read_csv(path)


### Funcao: read_optional_csv

Definicao isolada de read_optional_csv para reutilizacao nas etapas seguintes.


In [ ]:

def read_optional_csv(path: Path):
    if not path.exists():
        return None
    return pd.read_csv(path)


In [ ]:

benchmark_calibration_wide_df = read_required_csv(benchmark_calibration_path, "benchmark calibration wide")
benchmark_brier_wide_df = read_required_csv(benchmark_brier_path, "benchmark brier wide")
support_reference_df = read_required_csv(support_reference_path, "support reference")

p11_1_by_horizon_df = read_optional_csv(p11_1_by_horizon_path)
legacy_reliability_all_df = read_optional_csv(legacy_reliability_all_path)
legacy_reliability_h20_df = read_optional_csv(legacy_reliability_h20_path)

# ------------------------------
# 4) Reshape benchmark-wide tables to long format
# ------------------------------
id_cols = ["model_key", "display_name", "family"]

calibration_long = benchmark_calibration_wide_df.melt(
    id_vars=id_cols,
    var_name="metric_horizon",
    value_name="calibration_gap"
)
calibration_long["horizon_week"] = (
    calibration_long["metric_horizon"]
    .astype(str)
    .str.extract(r"h(\d+)", expand=False)
    .astype(float)
)
calibration_long = calibration_long.drop(columns=["metric_horizon"])

brier_long = benchmark_brier_wide_df.melt(
    id_vars=id_cols,
    var_name="metric_horizon",
    value_name="brier_value"
)
brier_long["horizon_week"] = (
    brier_long["metric_horizon"]
    .astype(str)
    .str.extract(r"h(\d+)", expand=False)
    .astype(float)
)
brier_long = brier_long.drop(columns=["metric_horizon"])

support_long = support_reference_df.copy()

support_long = (
    support_long.sort_values(["horizon_week", "n_evaluable_enrollments"], ascending=[True, False])
    .drop_duplicates(subset=["horizon_week"], keep="first")
    .reset_index(drop=True)
)

# ------------------------------
# 5) Merge core required pieces
# ------------------------------
audit_df = (
    calibration_long
    .merge(brier_long, on=["model_key", "display_name", "family", "horizon_week"], how="outer")
    .merge(support_long, on="horizon_week", how="left")
)

audit_df["metric_scope"] = "shared_benchmark_horizon"
audit_df["slope_available"] = False
audit_df["intercept_available"] = False
audit_df["approx_calibration_slope"] = np.nan
audit_df["approx_calibration_intercept"] = np.nan

# ------------------------------
# 6) Optionally enrich with P11.1 outputs if available
# ------------------------------
if p11_1_by_horizon_df is not None:
    p11 = p11_1_by_horizon_df.copy()

    rename_map = {}
    if "model_label" in p11.columns and "model_key" not in p11.columns:
        rename_map["model_label"] = "model_key"
    if "horizon" in p11.columns and "horizon_week" not in p11.columns:
        rename_map["horizon"] = "horizon_week"
    if "calibration_slope" in p11.columns and "approx_calibration_slope" not in p11.columns:
        rename_map["calibration_slope"] = "approx_calibration_slope"
    if "calibration_intercept" in p11.columns and "approx_calibration_intercept" not in p11.columns:
        rename_map["calibration_intercept"] = "approx_calibration_intercept"

    p11 = p11.rename(columns=rename_map)

    wanted = ["model_key", "horizon_week", "approx_calibration_slope", "approx_calibration_intercept"]
    available = [c for c in wanted if c in p11.columns]
    if set(["model_key", "horizon_week"]).issubset(available):
        p11 = p11[available].copy()
        audit_df = audit_df.merge(
            p11,
            on=["model_key", "horizon_week"],
            how="left",
            suffixes=("", "_p11")
        )

        if "approx_calibration_slope_p11" in audit_df.columns:
            audit_df["approx_calibration_slope"] = audit_df["approx_calibration_slope_p11"].combine_first(
                audit_df["approx_calibration_slope"]
            )
            audit_df = audit_df.drop(columns=["approx_calibration_slope_p11"])

        if "approx_calibration_intercept_p11" in audit_df.columns:
            audit_df["approx_calibration_intercept"] = audit_df["approx_calibration_intercept_p11"].combine_first(
                audit_df["approx_calibration_intercept"]
            )
            audit_df = audit_df.drop(columns=["approx_calibration_intercept_p11"])

        audit_df["slope_available"] = audit_df["approx_calibration_slope"].notna()
        audit_df["intercept_available"] = audit_df["approx_calibration_intercept"].notna()

# ------------------------------
# 7) Reliability bins output
# ------------------------------
if legacy_reliability_all_df is not None:
    reliability_bins_df = legacy_reliability_all_df.copy()
elif legacy_reliability_h20_df is not None:
    reliability_bins_df = legacy_reliability_h20_df.copy()
else:
    reliability_bins_df = pd.DataFrame({
        "note": ["No legacy reliability bin table was available at this stage."]
    })

# ------------------------------
# 8) Paper summary
# ------------------------------
paper_summary_df = (
    audit_df.groupby(["model_key", "display_name", "family"], dropna=False)
    .agg(
        mean_calibration_gap=("calibration_gap", "mean"),
        max_calibration_gap=("calibration_gap", "max"),
        mean_brier=("brier_value", "mean"),
        min_brier=("brier_value", "min"),
        n_horizons=("horizon_week", "nunique"),
        all_slope_available=("slope_available", "all"),
        all_intercept_available=("intercept_available", "all"),
    )
    .reset_index()
    .sort_values(["mean_calibration_gap", "mean_brier", "display_name"])
    .reset_index(drop=True)
)

# ------------------------------
# 9) Save outputs
# ------------------------------
audit_path = TABLES_DIR / "table_p26_5_all_tuned_calibration_audit.csv"
paper_summary_path = TABLES_DIR / "table_p26_5_paper_summary_all_tuned_models.csv"
reliability_bins_path = TABLES_DIR / "table_p26_5_reliability_bins_all_tuned_models.csv"

audit_df.to_csv(audit_path, index=False)
paper_summary_df.to_csv(paper_summary_path, index=False)
reliability_bins_df.to_csv(reliability_bins_path, index=False)

# ------------------------------
# 10) Output
# ------------------------------
print("\nUnified calibration audit:")
display(audit_df)

print("\nPaper summary across tuned models:")
display(paper_summary_df)

print("\nReliability bins artifact:")
display(reliability_bins_df.head(20))

print("\nOptional enrichment status:")
print(f"- P11.1 by horizon available: {p11_1_by_horizon_df is not None}")
print(f"- Legacy reliability bins (all) available: {legacy_reliability_all_df is not None}")
print(f"- Legacy reliability bins (h20) available: {legacy_reliability_h20_df is not None}")

print("\nSaved:")
print("-", audit_path.resolve())
print("-", paper_summary_path.resolve())
print("-", reliability_bins_path.resolve())


### D6.3 — Calibration Interpretation Summary for Paper Use


In [ ]:
# ==============================================================
# D6.3 — Calibration Interpretation Summary for Paper Use
# --------------------------------------------------------------
# Purpose:
#   Build a paper-oriented interpretation layer from the unified
#   expanded calibration audit across tuned benchmark models.
#
# Inputs expected from previous cells:
#   - TABLES_DIR
#   - table_p26_5_all_tuned_calibration_audit.csv
#   - table_p26_5_paper_summary_all_tuned_models.csv
#
# Outputs:
#   - table_p26_6_calibration_interpretation_summary.csv
#   - table_p26_6_horizon_leaders.csv
# ==============================================================

print("\n" + "=" * 70)
print("D6.3 — Calibration Interpretation Summary for Paper Use")
print("=" * 70)

from pathlib import Path
import pandas as pd

# ------------------------------
# 1) Basic checks
# ------------------------------
required_names = ["TABLES_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run the required upstream cells first."
    )

TABLES_DIR.mkdir(parents=True, exist_ok=True)

audit_path = TABLES_DIR / "table_p26_5_all_tuned_calibration_audit.csv"
paper_summary_path = TABLES_DIR / "table_p26_5_paper_summary_all_tuned_models.csv"

if not audit_path.exists():
    raise FileNotFoundError(f"Missing required file: {audit_path}")
if not paper_summary_path.exists():
    raise FileNotFoundError(f"Missing required file: {paper_summary_path}")

audit_df = pd.read_csv(audit_path)
paper_summary_df = pd.read_csv(paper_summary_path)

required_audit_cols = [
    "model_key",
    "display_name",
    "family",
    "horizon_week",
    "calibration_gap",
    "brier_value",
]
missing_audit_cols = [c for c in required_audit_cols if c not in audit_df.columns]
if missing_audit_cols:
    raise KeyError(
        "Unified calibration audit is missing required columns: "
        + ", ".join(missing_audit_cols)
    )

# ------------------------------
# 2) Horizon leaders
# ------------------------------
horizon_leader_rows = []

for horizon, g in audit_df.groupby("horizon_week"):
    g_sorted = g.sort_values(
        by=["calibration_gap", "brier_value", "display_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    best = g_sorted.iloc[0]

    horizon_leader_rows.append({
        "horizon_week": int(horizon),
        "best_model_key": best["model_key"],
        "best_display_name": best["display_name"],
        "best_family": best["family"],
        "best_calibration_gap": float(best["calibration_gap"]),
        "best_brier_value": float(best["brier_value"]),
        "n_models_compared": int(len(g_sorted)),
    })

horizon_leaders_df = pd.DataFrame(horizon_leader_rows).sort_values("horizon_week").reset_index(drop=True)

# ------------------------------
# 3) Narrative-ready interpretation summary
# ------------------------------
summary_rows = []

for _, row in paper_summary_df.iterrows():
    interp_parts = []

    mean_gap = row["mean_calibration_gap"]
    if pd.notna(mean_gap):
        if mean_gap <= 0.03:
            interp_parts.append("very strong average calibration")
        elif mean_gap <= 0.06:
            interp_parts.append("strong average calibration")
        elif mean_gap <= 0.10:
            interp_parts.append("moderate calibration")
        else:
            interp_parts.append("weaker calibration")

    max_gap = row["max_calibration_gap"]
    if pd.notna(max_gap):
        if max_gap <= 0.04:
            interp_parts.append("stable across horizons")
        elif max_gap <= 0.08:
            interp_parts.append("some horizon sensitivity")
        else:
            interp_parts.append("substantial horizon sensitivity")

    mean_brier = row["mean_brier"]
    if pd.notna(mean_brier):
        if mean_brier <= 0.13:
            interp_parts.append("strong overall horizon-level accuracy")
        elif mean_brier <= 0.17:
            interp_parts.append("competitive overall horizon-level accuracy")
        else:
            interp_parts.append("weaker overall horizon-level accuracy")

    if bool(row.get("all_slope_available", False)) and bool(row.get("all_intercept_available", False)):
        strengthening_status = "expanded slope/intercept diagnostics available"
    else:
        strengthening_status = "expanded slope/intercept diagnostics not yet available"

    summary_rows.append({
        "model_key": row["model_key"],
        "display_name": row["display_name"],
        "family": row["family"],
        "mean_calibration_gap": row["mean_calibration_gap"],
        "max_calibration_gap": row["max_calibration_gap"],
        "mean_brier": row["mean_brier"],
        "min_brier": row["min_brier"],
        "n_horizons": row["n_horizons"],
        "all_slope_available": row["all_slope_available"],
        "all_intercept_available": row["all_intercept_available"],
        "interpretation_summary": "; ".join(interp_parts),
        "strengthening_status": strengthening_status,
    })

interpretation_df = pd.DataFrame(summary_rows).sort_values(
    ["mean_calibration_gap", "mean_brier", "display_name"]
).reset_index(drop=True)

# ------------------------------
# 4) Save outputs
# ------------------------------
interpretation_path = TABLES_DIR / "table_p26_6_calibration_interpretation_summary.csv"
horizon_leaders_path = TABLES_DIR / "table_p26_6_horizon_leaders.csv"

interpretation_df.to_csv(interpretation_path, index=False)
horizon_leaders_df.to_csv(horizon_leaders_path, index=False)

# ------------------------------
# 5) Output
# ------------------------------
print("\nCalibration interpretation summary:")
display(interpretation_df)

print("\nHorizon leaders:")
display(horizon_leaders_df)

print("\nSaved:")
print("-", interpretation_path.resolve())
print("-", horizon_leaders_path.resolve())